# 15 · Deployment and Model Optimization

**Part VII — Engineering, trust and practice** · Estimated time: ~120 min

A trained model is a file of numbers; a *deployed* model is a service that answers requests within a
latency budget, on a cost budget, reliably, for months, while the world around it changes. This
notebook covers the road from the first to the second. We begin with what changes when a model stops
training and starts answering (evaluation mode, inference mode, batching, the latency/throughput
trade-off) and build a benchmarking helper that reports percentiles rather than averages. We then look
at how to save models so that they can be versioned and reloaded years later, and at the three export
routes that decouple a model from the Python code that trained it: `torch.export`, TorchScript and
ONNX, the last of which we run with ONNX Runtime and verify numerically. The middle of the notebook is
about making models smaller and faster: quantization (derived from scale and zero-point up to
quantization-aware training), pruning, and knowledge distillation. We end with serving (a FastAPI
service and a Dockerfile, written to disk and explained) and with the operational side of models:
monitoring, drift, rollback and model cards.

## Learning objectives

- Benchmark inference correctly (warm-up, synchronization, percentiles) and explain the latency–throughput trade-off of batching.
- Save, version and reload models with their preprocessing, and explain the roles of `state_dict`, full pickling and safetensors.
- Export a model with `torch.export`, TorchScript and ONNX, run it with ONNX Runtime and verify numerical agreement.
- Derive affine int8 quantization, apply dynamic quantization, and implement fake quantization with a straight-through estimator for quantization-aware training.
- Prune a network with `torch.nn.utils.prune`, measure the sparsity–accuracy curve, and explain why unstructured sparsity rarely speeds up dense hardware.
- Implement knowledge distillation with soft targets and temperature and compare it with training from scratch.
- Write an inference service (FastAPI + ONNX Runtime) and a container definition, and describe the serving landscape (TorchServe, Triton, vLLM, edge runtimes).
- Monitor a deployed model for drift, plan A/B tests and rollbacks, and write a model card.

## Prerequisites

- Notebook 05 · The PyTorch Workflow and Best Practices (checkpoints, `state_dict`, evaluation loops).
- Notebook 06 · Convolutional Neural Networks (the CNN architecture used as the running example).
- Notebook 14 · Efficient Training and Scaling (timing with synchronization, memory arithmetic, mixed precision).

## Contents

1. From training to inference
2. Saving and versioning models
3. Export: `torch.export`, TorchScript, ONNX and ONNX Runtime
4. Quantization
5. Pruning
6. Knowledge distillation
7. Serving
8. Operating models: monitoring, drift, rollback and model cards
- Exercises · Summary · Further reading

In [ ]:
import copy, hashlib, io, json, math, os, platform, random, time, textwrap, warnings
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = DATA_DIR / "models_nb15"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

# Optional libraries: the notebook explains and skips gracefully when they are missing.
try:
    import onnx
    HAVE_ONNX = True
    print(f"onnx {onnx.__version__} available")
except ImportError:
    HAVE_ONNX = False
    print("onnx is not installed: ONNX inspection cells will be skipped (see environment.yml).")
try:
    import onnxruntime as ort
    HAVE_ORT = True
    print(f"onnxruntime {ort.__version__} available | providers: {ort.get_available_providers()}")
except ImportError:
    HAVE_ORT = False
    print("onnxruntime is not installed: ONNX Runtime cells will be skipped (see environment.yml).")

## 1. From training to inference

### 1.1 What changes at inference time

Training and inference execute the same forward function, but almost everything around it differs:

- **Mode.** `model.eval()` switches layers with train/test behaviour: dropout becomes the identity
  and BatchNorm uses its running statistics instead of batch statistics. Forgetting it is the most
  common deployment bug: predictions become stochastic (dropout) or depend on the other items in the
  batch (BatchNorm), and a request served alone gets a different answer than the same request served
  in a batch.
- **No autograd.** Inside `torch.no_grad()` no computational graph is built, which saves the memory
  of every intermediate tensor and some time. `torch.inference_mode()` goes further: tensors created
  inside it carry no version counters or view metadata for autograd at all, which is slightly faster
  again, at the price that such tensors cannot later be used in autograd computations. For serving,
  `inference_mode` is the right choice.
- **Batching.** Training uses large batches for throughput; inference requests arrive one at a time.
  Serving a batch of one wastes most of the hardware, so servers collect requests for a few
  milliseconds and run them together, which is the **latency versus throughput** trade-off: batching
  raises throughput (requests per second) and raises the latency of each request (it waits for the
  batch to fill and then shares the compute).
- **Percentiles, not means.** A user notices the slowest requests. Service-level objectives are stated
  as "p99 latency below 50 ms", i.e. 99 % of requests must finish within 50 ms, because a mean hides
  the tail caused by garbage collection, cache misses, contention and occasional large batches.

The helper below is the inference counterpart of the timing helper of Notebook 14 · Efficient Training
and Scaling: it warms up, times
each call individually with proper synchronization, and returns percentiles in milliseconds.

In [ ]:
def sync():
    """Wait for queued device work (a no-op on the CPU)."""
    if device.type == "cuda":
        torch.cuda.synchronize()


def benchmark_latency(fn, n_warmup=10, n_iters=50):
    """Latency statistics (milliseconds) of fn() over n_iters calls after n_warmup warm-up calls."""
    for _ in range(n_warmup):
        fn()
    sync()
    times = []
    for _ in range(n_iters):
        t0 = time.perf_counter()
        fn()
        sync()
        times.append((time.perf_counter() - t0) * 1e3)
    t = np.array(times)
    return {"mean": float(t.mean()), "p50": float(np.percentile(t, 50)), "p90": float(np.percentile(t, 90)),
            "p99": float(np.percentile(t, 99)), "min": float(t.min())}


def fmt_latency(stats):
    return f"mean {stats['mean']:7.3f} ms | p50 {stats['p50']:7.3f} | p90 {stats['p90']:7.3f} | p99 {stats['p99']:7.3f}"


# a first, model-free use: how long does a 512x512 matmul take on this device?
a = torch.randn(512, 512, device=device)
print("512x512 matmul:", fmt_latency(benchmark_latency(lambda: a @ a)))

Note the gap between p50 and p99 even for a fixed-size matmul on an idle machine: that tail is what a
serving system has to design around, and it is why load tests report distributions.

### 1.2 The running example: a CNN on MNIST

Everything in this notebook is demonstrated on one small convolutional classifier trained on MNIST,
which is large enough to have interesting structure (two convolutions, two linear layers, dropout)
and small enough to be exported, quantized, pruned and distilled in seconds. As in Notebook 14 we load
the dataset into memory tensors. We keep two versions of the images: `X_*_raw` with pixel values in
$[0, 1]$, which is what a client would send to a service, and `X_*` normalized with the MNIST mean and
standard deviation, which is what the classifier consumes. Keeping the distinction explicit is the
first step of "storing the preprocessing with the model" (Section 2).

In [ ]:
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081


def load_mnist(train, n=None):
    ds = datasets.MNIST(DATA_DIR, train=train, download=True)
    X = ds.data.float().div(255.0).unsqueeze(1)      # (N, 1, 28, 28) in [0, 1]
    y = ds.targets.clone()
    if n is not None:
        X, y = X[:n], y[:n]
    return X, y


N_TRAIN = 8_000 if FAST_RUN else 60_000
N_TEST = 2_000 if FAST_RUN else 10_000
X_train_raw, y_train = load_mnist(True, N_TRAIN)
X_test_raw, y_test = load_mnist(False, N_TEST)
X_train, X_test = (X_train_raw - MNIST_MEAN) / MNIST_STD, (X_test_raw - MNIST_MEAN) / MNIST_STD
X_train_raw, y_train, X_test_raw, y_test, X_train, X_test = [
    t.to(device) for t in (X_train_raw, y_train, X_test_raw, y_test, X_train, X_test)]
print("train", tuple(X_train.shape), "| test", tuple(X_test.shape))


class MNISTCNN(nn.Module):
    """conv(1->c1) -> pool -> conv(c1->c2) -> pool -> fc(hidden) -> dropout -> fc(10); input is normalized."""

    def __init__(self, c1=16, c2=32, hidden=128, p_drop=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(1, c1, 3, padding=1)
        self.conv2 = nn.Conv2d(c1, c2, 3, padding=1)
        self.fc1 = nn.Linear(c2 * 7 * 7, hidden)
        self.drop = nn.Dropout(p_drop)
        self.fc2 = nn.Linear(hidden, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.relu(self.fc1(x.flatten(1)))
        return self.fc2(self.drop(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def iterate_minibatches(X, y, batch_size, shuffle=True):
    n = X.shape[0]
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    for start in range(0, n, batch_size):
        j = idx[start:start + batch_size]
        yield X[j], y[j]


def train_one_epoch(model, opt, X, y, batch_size=64, loss_fn=None):
    """One epoch; loss_fn(logits, yb, idx) defaults to cross-entropy (idx = indices, used for distillation)."""
    model.train()
    loss_sum = torch.zeros((), device=X.device)
    n = X.shape[0]
    perm = torch.randperm(n, device=X.device)
    for start in range(0, n, batch_size):
        idx = perm[start:start + batch_size]
        logits = model(X[idx])
        loss = F.cross_entropy(logits, y[idx]) if loss_fn is None else loss_fn(logits, y[idx], idx)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        loss_sum += loss.detach() * idx.numel()
    return loss_sum.item() / n


@torch.inference_mode()
def evaluate(model, X, y, batch_size=500):
    """Mean cross-entropy and accuracy; the model may live on any device, X and y follow it."""
    model.eval()
    params = list(model.parameters())
    dev = params[0].device if params else torch.device("cpu")     # quantized modules may own no Parameters
    loss_sum, correct = 0.0, 0
    for start in range(0, X.shape[0], batch_size):
        xb, yb = X[start:start + batch_size].to(dev), y[start:start + batch_size].to(dev)
        logits = model(xb)
        loss_sum += F.cross_entropy(logits, yb, reduction="sum").item()
        correct += (logits.argmax(1) == yb).sum().item()
    return loss_sum / X.shape[0], correct / X.shape[0]


EPOCHS = 2 if FAST_RUN else 4
torch.manual_seed(SEED)
model = MNISTCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
print(f"MNISTCNN: {count_params(model):,} parameters")
for epoch in range(EPOCHS):
    t0 = time.perf_counter()
    tr = train_one_epoch(model, opt, X_train, y_train)
    sched.step()
    te_loss, te_acc = evaluate(model, X_test, y_test)
    print(f"epoch {epoch + 1} | train loss {tr:.4f} | test loss {te_loss:.4f} | test acc {te_acc:.4f} | {time.perf_counter() - t0:.1f}s")
BASE_ACC = te_acc

The model reaches about 97–98 % test accuracy in the fast setting and over 99 % with the full data
and four epochs. It is our reference: every optimization below is judged by how much of this accuracy
it keeps and how much size or latency it saves. Let us first make the `eval()` point concrete.

In [ ]:
x1 = X_test[:1]
model.train()
with torch.no_grad():
    out_a, out_b = model(x1), model(x1)
print("train mode: two forward passes of the same input differ by", (out_a - out_b).abs().max().item(), "(dropout)")
model.eval()
with torch.inference_mode():
    out_c, out_d = model(x1), model(x1)
print("eval mode : two forward passes differ by", (out_c - out_d).abs().max().item())

stats_autograd = benchmark_latency(lambda: model(X_test[:64]))             # autograd graph is built
with torch.no_grad():
    stats_nograd = benchmark_latency(lambda: model(X_test[:64]))
with torch.inference_mode():
    stats_inf = benchmark_latency(lambda: model(X_test[:64]))
print("batch 64 forward, eval mode:")
print("  with autograd      ", fmt_latency(stats_autograd))
print("  torch.no_grad      ", fmt_latency(stats_nograd))
print("  torch.inference_mode", fmt_latency(stats_inf))

In training mode the two passes differ because dropout samples a fresh mask each time; in evaluation
mode they are identical. The three timings show that recording the autograd graph costs a measurable
fraction of a small forward pass and that `inference_mode` shaves a little more; for a large model the
relative gain shrinks, but the memory saving of not keeping activations remains.

### 1.3 Latency versus throughput

We now measure the forward pass for batch sizes from 1 to 256 and convert the results into the two
quantities a service cares about: the latency of one call and the throughput in images per second.

In [ ]:
model.eval()
batch_sizes = [1, 4, 16, 64, 256]
lat_rows = []
with torch.inference_mode():
    for bs in batch_sizes:
        xb = X_test[:bs]
        st = benchmark_latency(lambda: model(xb), n_warmup=5, n_iters=30)
        lat_rows.append((bs, st))
        print(f"batch {bs:4d} | {fmt_latency(st)} | per image {st['p50'] / bs * 1e3:7.1f} µs | "
              f"throughput {bs / st['p50'] * 1e3:9.0f} img/s")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(batch_sizes, [st["p50"] for _, st in lat_rows], "o-", label="p50")
axes[0].plot(batch_sizes, [st["p99"] for _, st in lat_rows], "s--", label="p99")
axes[0].set_xscale("log", base=2); axes[0].set_yscale("log")
axes[0].set_xlabel("batch size"); axes[0].set_ylabel("latency per call (ms)"); axes[0].set_title("Latency"); axes[0].legend()
axes[1].plot(batch_sizes, [bs / st["p50"] * 1e3 for bs, st in lat_rows], "o-")
axes[1].set_xscale("log", base=2)
axes[1].set_xlabel("batch size"); axes[1].set_ylabel("images / s"); axes[1].set_title("Throughput")
plt.tight_layout(); plt.show()
PYTORCH_LAT_B1 = lat_rows[0][1]["p50"]

The latency curve is flat at small batch sizes (a fixed per-call overhead of kernel launches and
Python dominates) and becomes linear at large ones (the arithmetic dominates); the throughput curve is
the mirror image, rising steeply and then saturating. The knee of the curve is the batch size at which
the hardware is busy, and a serving system aims to operate near it: below the knee it wastes compute,
above it adds latency for no throughput gain. On a GPU the knee sits at a much larger batch (hundreds
of images for this model) than on a CPU (a few dozen), which is why GPU serving depends so much on
batching requests.

> **Key idea.** Measure inference as a distribution (p50/p99) over batch sizes; the batch size at
> the knee of the throughput curve is the one to serve at, and the request batcher's job is to reach
> it without waiting too long.

## 2. Saving and versioning models

### 2.1 `state_dict`, whole-model pickling and safetensors

Notebook 05 · The PyTorch Workflow and Best Practices introduced `model.state_dict()`, the ordered
dictionary of parameter and buffer tensors.
It is the right unit of persistence because it contains only data: loading it requires the class that
defines the architecture, which lives in versioned code, and no arbitrary Python objects are
unpickled. `torch.save(model, path)` instead pickles the whole module object, including a *reference*
to its class by module path; loading it needs the same class importable under the same name and, since
pickle can execute arbitrary code, it is a security risk with files from untrusted sources. PyTorch 2.6
made `weights_only=True` the default of `torch.load`, which restricts unpickling to tensors, primitive
containers and a small allow-list; a full-model file then fails to load unless you explicitly pass
`weights_only=False`. Refusing arbitrary objects by default is exactly the behaviour a deployment
pipeline wants: only `state_dict`s and plain metadata should ever be loaded from disk.

**safetensors** (the format used by the Hugging Face ecosystem) takes this to its conclusion: a file
is a JSON header describing tensor names, dtypes and shapes followed by the raw bytes, with no code
execution possible on load, memory-mapped zero-copy loading, and lazy loading of individual tensors.
It is the format to prefer for sharing weights; we do not depend on the library here, but the bundle
we build below has the same "data only" property.

In [ ]:
sd_path = MODEL_DIR / "mnist_cnn_state_dict.pt"
torch.save(model.state_dict(), sd_path)

reloaded = MNISTCNN().to(device)
reloaded.load_state_dict(torch.load(sd_path, map_location=device, weights_only=True))
reloaded.eval()
with torch.inference_mode():
    diff = (reloaded(X_test[:64]) - model(X_test[:64])).abs().max().item()
print(f"state_dict round trip: {sd_path.stat().st_size / 1024:.1f} KB, max output difference {diff:.1e}")
print("keys:", list(model.state_dict().keys()))

full_path = MODEL_DIR / "mnist_cnn_full_pickle.pt"
torch.save(model, full_path)
try:
    _ = torch.load(full_path, map_location=device)                  # default weights_only=True in PyTorch >= 2.6
    print("torch.load(full model) with the default settings succeeded (older PyTorch default).")
except Exception as e:
    print(f"torch.load(full model) with default settings raised {type(e).__name__}: the pickle contains a class, "
          "which weights_only=True refuses.")
full_model = torch.load(full_path, map_location=device, weights_only=False)
print("with weights_only=False:", type(full_model).__name__, "loaded; it requires the MNISTCNN class to be importable.")

### 2.2 A versioned model bundle with preprocessing and provenance

A `state_dict` alone is not a deployable artifact. To reproduce a prediction months later we also
need the architecture configuration (so that the right class with the right widths is built), the
preprocessing (the exact normalization constants; a mismatch here silently ruins accuracy), the
training data version and metrics, the framework versions, and an integrity check. The bundle below
stores all of that in one file using only tensors, numbers and strings, so it loads under
`weights_only=True`. We adopt **semantic versioning** for models:

- **MAJOR** changes the contract with clients: a different input format, output classes or
  preprocessing. Clients must change.
- **MINOR** improves the model within the same contract: retraining on more data, a better
  architecture with the same inputs and outputs. Clients need not change, but should re-validate.
- **PATCH** changes nothing about predictions: metadata fixes, re-exports, re-packaging.

A **model registry** (MLflow's registry, or a database table plus object storage) records every
version with its bundle location, metrics, lineage (data version, code commit, training config) and a
*stage* such as `staging`, `production` or `archived`, and is the place where "which model is serving
right now?" has a single answer. The registry entry is what promotion, A/B tests and rollbacks
(Section 8) manipulate.

In [ ]:
def sha256_of_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def save_bundle(model, version, metrics, path, notes=""):
    bundle = {
        "model_name": "mnist_cnn",
        "version": version,
        "architecture": {"class": "MNISTCNN", "c1": model.conv1.out_channels, "c2": model.conv2.out_channels,
                         "hidden": model.fc1.out_features, "p_drop": model.drop.p},
        "preprocessing": {"input": "float32 (N, 1, 28, 28) with pixel values in [0, 1]",
                          "normalize_mean": MNIST_MEAN, "normalize_std": MNIST_STD},
        "outputs": "logits over classes 0-9 (apply softmax for probabilities)",
        "metrics": metrics,
        "training": {"dataset": "MNIST (torchvision)", "n_train": N_TRAIN, "epochs": EPOCHS, "seed": SEED, "fast_run": FAST_RUN},
        # str(...) matters: torch.__version__ is a TorchVersion object, which the safe
        # weights_only loader would refuse to unpickle
        "environment": {"torch": str(torch.__version__), "python": platform.python_version(), "device": str(device)},
        "created": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "notes": notes,
        "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    }
    torch.save(bundle, path)
    return sha256_of_file(path)


def load_bundle(path, expected_sha=None):
    if expected_sha is not None and sha256_of_file(path) != expected_sha:
        raise ValueError("bundle hash mismatch: file corrupted or tampered with")
    bundle = torch.load(path, map_location="cpu", weights_only=True)
    arch = bundle["architecture"]
    m = MNISTCNN(arch["c1"], arch["c2"], arch["hidden"], arch["p_drop"])
    m.load_state_dict(bundle["state_dict"])
    return m.eval(), bundle


bundle_path = MODEL_DIR / "mnist_cnn-1.0.0.pt"
metrics = {"test_accuracy": round(BASE_ACC, 4), "test_loss": round(te_loss, 4), "latency_p50_ms_batch1": round(PYTORCH_LAT_B1, 3)}
sha = save_bundle(model, "1.0.0", metrics, bundle_path, notes="baseline CNN, cosine schedule, Adam 1e-3")
model_v1, meta = load_bundle(bundle_path, expected_sha=sha)
print(f"saved {bundle_path.name} ({bundle_path.stat().st_size / 1024:.1f} KB) sha256={sha[:16]}...")
print(json.dumps({k: v for k, v in meta.items() if k != "state_dict"}, indent=1)[:900])
print("reloaded accuracy:", round(evaluate(model_v1, X_test.cpu(), y_test.cpu())[1], 4))

The bundle reproduces the test accuracy on reload, carries the preprocessing constants that a client
or a serving wrapper must apply, and is protected by its hash. The manifest is what a registry stores
next to the file. The final ingredient of "storing the preprocessing with the model" is to put the
preprocessing *inside the computational graph*, so that an exported model consumes raw client data
and cannot be used with the wrong normalization. The `ServingModel` wrapper below does this: it takes
pixels in $[0, 1]$, normalizes with buffers stored in the module, runs the classifier and returns
probabilities. All exports in Section 3 are exports of this wrapper.

In [ ]:
class ServingModel(nn.Module):
    """Preprocessing + classifier + softmax in one module: raw pixels in [0, 1] -> class probabilities."""

    def __init__(self, classifier, mean=MNIST_MEAN, std=MNIST_STD):
        super().__init__()
        self.classifier = classifier
        self.register_buffer("mean", torch.tensor(float(mean)))
        self.register_buffer("std", torch.tensor(float(std)))

    def forward(self, pixels):
        x = (pixels - self.mean) / self.std
        return F.softmax(self.classifier(x), dim=1)


serving = ServingModel(copy.deepcopy(model)).cpu().eval()        # exports are done on the CPU copy
example = X_test_raw[:8].cpu().contiguous()                       # example input: batch of 8 raw images
with torch.inference_mode():
    probs_ref = serving(example)
print("serving model output:", tuple(probs_ref.shape), "| rows sum to", probs_ref.sum(1)[:3].tolist())
print("predicted:", probs_ref.argmax(1).tolist(), "| true:", y_test[:8].tolist())

## 3. Export: `torch.export`, TorchScript, ONNX and ONNX Runtime

### 3.1 Why export at all

A PyTorch module is Python code plus tensors. Running it requires a Python interpreter, the exact
class definition, and the whole PyTorch library, and it executes eagerly, one operator at a time. None
of that is acceptable on a phone, inside a C++ game engine, in a browser, or on a fleet of servers
where a Python dependency is an operational liability. **Exporting** turns the module into a
self-contained *graph* of operators with known shapes and dtypes that a runtime can load without the
original code. PyTorch offers three routes with different maturity and targets:

| route | how the graph is captured | output | status |
|---|---|---|---|
| `torch.export` | TorchDynamo traces the bytecode, sound (guards on shapes) | `ExportedProgram` (ATen IR), `.pt2` file | the modern standard, used by ExecuTorch, AOTInductor and the ONNX exporter |
| TorchScript | tracing (record ops) or scripting (compile a Python subset) | `ScriptModule`, `.pt` file for libtorch | maintenance mode; still used by older C++ deployments and TorchServe |
| ONNX | `torch.export` (dynamo) or TorchScript (legacy) then translation to ONNX ops | `.onnx` protobuf | the interoperable format for ONNX Runtime, TensorRT, OpenVINO, CoreML converters |

The output of any export is verified the same way: run the original and the exported artifact on the
same inputs and compare numerically, and do it for more than one batch size if the batch dimension
is supposed to be dynamic.

### 3.2 `torch.export`

`torch.export.export(module, args, dynamic_shapes=...)` returns an `ExportedProgram`: a functional
graph of ATen operators together with the parameters and buffers it needs and the **guards** under
which the trace is valid. A shape marked with `torch.export.Dim` becomes a symbol; everything else is
specialized to the example input's value. One subtlety worth knowing is **0/1 specialization**:
the exporter treats dimension values 0 and 1 as special because they change broadcasting behaviour,
so a batch dimension exported from an example with batch 1 would be static. We therefore use an example
batch of 8 and declare the batch dimension dynamic. Depending on the PyTorch version, a lower bound of
1 on a dynamic dimension may be rejected in favour of 2; the cell handles both.

In [ ]:
EXPORT_OK = False
try:
    from torch.export import export as torch_export, Dim

    def export_with_min_batch(min_batch):
        batch = Dim("batch", min=min_batch, max=4096)
        return torch_export(serving, (example,), dynamic_shapes={"pixels": {0: batch}})

    try:
        ep = export_with_min_batch(1)
    except Exception as e_min1:
        print(f"export with min=1 was rejected ({type(e_min1).__name__}); retrying with min=2 "
              "(this PyTorch version specializes the dimension values 0 and 1)")
        ep = export_with_min_batch(2)
    EXPORT_OK = True

    graph_lines = str(ep.graph).splitlines()
    print(f"ExportedProgram with {len(list(ep.graph.nodes))} nodes; first lines of the graph:")
    print("\n".join(graph_lines[:14]))
    print("...")
    with torch.inference_mode():
        out8 = ep.module()(example)
        out3 = ep.module()(X_test_raw[:3].cpu().contiguous())
    print(f"\nmax |exported - eager| at batch 8: {(out8 - probs_ref).abs().max().item():.2e} | batch 3 output shape {tuple(out3.shape)}")

    pt2_path = MODEL_DIR / "mnist_cnn.pt2"
    torch.export.save(ep, str(pt2_path))
    ep_loaded = torch.export.load(str(pt2_path))
    with torch.inference_mode():
        out_loaded = ep_loaded.module()(example)
    print(f"saved {pt2_path.name} ({pt2_path.stat().st_size / 1024:.1f} KB); reloaded program agrees to "
          f"{(out_loaded - probs_ref).abs().max().item():.1e}")
    try:
        with torch.inference_mode():
            _ = ep.module()(example[:1])
        print("batch size 1 is accepted by this exported program.")
    except Exception as e1:
        print(f"batch size 1 is rejected by the shape guards ({type(e1).__name__}): the dynamic dimension has a lower bound of 2 "
              "in this version; serve single requests by padding to 2 or export with a static batch of 1 as well.")
except Exception as e:
    print(f"torch.export is unavailable or failed here: {type(e).__name__}: {str(e)[:200]}")

The graph printout shows the flavour of the ATen IR: placeholders for the input and for every
parameter and buffer, then one node per operator (`aten.sub`, `aten.div`, `aten.convolution`,
`aten.relu`, `aten.max_pool2d`, ..., `aten._softmax`), with the dropout already gone because the
module was exported in evaluation mode. The exported and the eager outputs agree to float rounding,
the program runs at a batch size it never saw, and it survives a save/load round trip without the
`ServingModel` class being needed. This artifact is the input of the whole PyTorch 2 deployment
stack: **AOTInductor** compiles it to a shared library callable from C++ without Python, and
**ExecuTorch** lowers it for mobile and embedded targets.

### 3.3 TorchScript: trace versus script

TorchScript predates `torch.export` and is still what many production C++ services and TorchServe
handlers load. It has two capture modes that behave very differently on control flow:

- **Tracing** (`torch.jit.trace`) runs the module on an example input and records the operators that
  executed. Anything that depends on the *values* of tensors, such as an `if` on `x.sum() > 0`, is
  evaluated once during tracing and baked in: the trace contains only the branch that was taken.
- **Scripting** (`torch.jit.script`) compiles the Python source of `forward` into TorchScript, a
  statically typed subset of Python, and preserves control flow. It fails on Python it does not
  understand (many library calls, dynamic typing).

The tiny module below makes the difference visible, after which we trace the serving model, save it
and verify it.

In [ ]:
class SignDependent(nn.Module):
    """Data-dependent control flow: tracing records only the branch taken for the example input."""

    def forward(self, x):
        if x.sum() > 0:
            return x * 2
        return -x


TS_OK = False
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        traced_cf = torch.jit.trace(SignDependent(), torch.ones(3))
        scripted_cf = torch.jit.script(SignDependent())
    neg = -torch.ones(3)
    print("eager   :", SignDependent()(neg).tolist())
    print("traced  :", traced_cf(neg).tolist(), "  <- wrong: the positive branch was baked in at trace time")
    print("scripted:", scripted_cf(neg).tolist(), "  <- correct: the if statement was compiled")
    print("\nscripted forward:\n" + textwrap.indent(scripted_cf.code, "    "))

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        traced_serving = torch.jit.trace(serving, example)
    ts_path = MODEL_DIR / "mnist_cnn_traced.pt"
    traced_serving.save(str(ts_path))
    ts_loaded = torch.jit.load(str(ts_path))
    with torch.inference_mode():
        out_ts = ts_loaded(example)
        out_ts3 = ts_loaded(X_test_raw[:3].cpu().contiguous())
    print(f"traced serving model: {ts_path.stat().st_size / 1024:.1f} KB | max diff vs eager {(out_ts - probs_ref).abs().max().item():.1e} "
          f"| batch 3 works: {tuple(out_ts3.shape)}")
    TS_OK = True
except Exception as e:
    print("TorchScript is unavailable in this build:", type(e).__name__, str(e)[:200])

The traced serving model works at other batch sizes because nothing in it depends on the batch value,
but tracing offers no guarantee of that: a `.view(8, -1)` written with the example batch size hard-coded
would silently produce a model that only works for 8 images. This is the fundamental weakness that
`torch.export`'s guards fix, and the reason the PyTorch team recommends `torch.export` for new
deployments while keeping TorchScript in maintenance.

### 3.4 ONNX export and ONNX Runtime

**ONNX** (Open Neural Network Exchange) is a vendor-neutral graph format: a protobuf file listing
operators from a versioned standard ("opset"), their attributes, the tensor initializers, and typed
inputs and outputs with optional symbolic dimensions. Its value is the ecosystem of runtimes that
consume it: **ONNX Runtime** (CPU, CUDA, TensorRT, DirectML, WebAssembly, mobile), TensorRT itself,
OpenVINO, and converters to CoreML. PyTorch has two exporters. The modern one is built on
`torch.export` (`dynamo=True`) and translates the ATen graph with the `onnxscript` library; the legacy
one is built on TorchScript tracing (`dynamo=False`) and is deprecated. We try the modern exporter
first, with the same dynamic batch dimension as above, and fall back if it is unavailable.

In [ ]:
onnx_path = MODEL_DIR / "mnist_cnn.onnx"
ONNX_OK, exporter_used = False, None
common = dict(input_names=["pixels"], output_names=["probs"])
attempts = [
    ("dynamo exporter, dynamic batch (min 1)", lambda: torch.onnx.export(
        serving, (example,), str(onnx_path), dynamo=True, external_data=False,
        dynamic_shapes={"pixels": {0: torch.export.Dim("batch", min=1, max=4096)}}, **common)),
    ("dynamo exporter, dynamic batch (min 2)", lambda: torch.onnx.export(
        serving, (example,), str(onnx_path), dynamo=True, external_data=False,
        dynamic_shapes={"pixels": {0: torch.export.Dim("batch", min=2, max=4096)}}, **common)),
    ("legacy TorchScript exporter", lambda: torch.onnx.export(
        serving, (example,), str(onnx_path), dynamo=False, opset_version=17,
        dynamic_axes={"pixels": {0: "batch"}, "probs": {0: "batch"}}, **common)),
]
for name, attempt in attempts:
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            attempt()
        exporter_used = name
        ONNX_OK = onnx_path.exists()
        break
    except Exception as e:
        print(f"{name} failed: {type(e).__name__}: {str(e)[:120]}")
if ONNX_OK:
    print(f"exported with the {exporter_used}: {onnx_path.name}, {onnx_path.stat().st_size / 1024:.1f} KB")
else:
    print("ONNX export is not possible in this environment; the ONNX cells below will be skipped.")

With the `onnx` package we can inspect the artifact: check it against the standard, list its operators
and its inputs and outputs with their (symbolic) shapes, and count the stored weights. The operator
list is worth comparing with the ATen graph above: the normalization became `Sub`/`Div` (or a fused
`Mul`/`Add`), convolutions became `Conv`, the linear layers `Gemm`, and softmax `Softmax`.

In [ ]:
if HAVE_ONNX and ONNX_OK:
    try:
        onnx_model = onnx.load(str(onnx_path))
        onnx.checker.check_model(onnx_model)
        g = onnx_model.graph
        shapes = lambda vi: [d.dim_param or d.dim_value for d in vi.type.tensor_type.shape.dim]
        print("IR version", onnx_model.ir_version, "| opsets", {imp.domain or "ai.onnx": imp.version for imp in onnx_model.opset_import})
        print("inputs :", [(i.name, shapes(i)) for i in g.input])
        print("outputs:", [(o.name, shapes(o)) for o in g.output])
        print("operators:", dict(Counter(n.op_type for n in g.node)))
        n_init = sum(int(np.prod(list(t.dims))) for t in g.initializer)
        print(f"initializers: {len(g.initializer)} tensors with {n_init:,} values "
              f"({n_init * 4 / 1024:.1f} KB as float32) vs {count_params(model):,} model parameters")
    except Exception as e:
        print("onnx inspection failed:", type(e).__name__, str(e)[:200])
else:
    print("onnx package or exported model unavailable; skipping inspection.")

Now the part that matters for deployment: running the model with ONNX Runtime, checking that its
outputs agree with PyTorch, measuring its accuracy on the test set, and comparing its latency with
eager PyTorch on the same CPU. ONNX Runtime applies graph optimizations at session creation (constant
folding, operator fusion such as Conv+ReLU) and uses its own thread pool and kernels, so it is often
faster than eager PyTorch for small models, where PyTorch's per-operator overhead dominates.

In [ ]:
ORT_OK = False


def ort_accuracy(sess, X_raw, y, batch=500):
    name = sess.get_inputs()[0].name
    X_np = X_raw.cpu().numpy().astype(np.float32)
    y_np = np.asarray(y.cpu())
    correct = 0
    for s in range(0, len(X_np), batch):
        out = sess.run(None, {name: X_np[s:s + batch]})[0]
        correct += int((out.argmax(1) == y_np[s:s + batch]).sum())
    return correct / len(X_np)


if HAVE_ORT and ONNX_OK:
    try:
        sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        in_name = sess.get_inputs()[0].name
        x_np = example.cpu().numpy().astype(np.float32)
        probs_ort = sess.run(None, {in_name: x_np})[0]
        ref_np = probs_ref.cpu().numpy().astype(np.float32)
        ort_maxdiff = float(np.abs(probs_ort - ref_np).max())
        print(f"max |ORT - PyTorch| on the example batch: {ort_maxdiff:.2e}")
        np.testing.assert_allclose(probs_ort, ref_np, rtol=1e-3, atol=1e-5)
        print("numerical verification passed (rtol 1e-3, atol 1e-5)")
        acc_ort = ort_accuracy(sess, X_test_raw, y_test)
        print(f"test accuracy: PyTorch {BASE_ACC:.4f} | ONNX Runtime {acc_ort:.4f}")

        print("\nCPU latency, eager PyTorch vs ONNX Runtime:")
        ort_lat = {}
        for bs in (1, 64):
            xb_cpu = X_test_raw[:bs].cpu().contiguous()
            xb_np = np.asarray(xb_cpu, dtype=np.float32)
            with torch.inference_mode():
                st_torch = benchmark_latency(lambda: serving(xb_cpu), n_warmup=5, n_iters=30)
            st_ort = benchmark_latency(lambda: sess.run(None, {in_name: xb_np}), n_warmup=5, n_iters=30)
            ort_lat[bs] = st_ort["p50"]
            print(f"  batch {bs:3d}: PyTorch p50 {st_torch['p50']:7.3f} ms (p99 {st_torch['p99']:7.3f}) | "
                  f"ORT p50 {st_ort['p50']:7.3f} ms (p99 {st_ort['p99']:7.3f}) | speed-up {st_torch['p50'] / st_ort['p50']:.2f}x")
        ORT_OK = True
    except Exception as e:
        print("ONNX Runtime failed:", type(e).__name__, str(e)[:200])
else:
    print("onnxruntime or the exported model is unavailable; skipping.")

The outputs agree to about $10^{-6}$ (differences come from different accumulation orders in the
convolution kernels), the accuracy is identical, and ONNX Runtime is typically 1.5–4× faster than
eager PyTorch at batch 1 on a CPU for a model of this size, with a smaller advantage at batch 64 where
the arithmetic dominates. These are the kinds of numbers that justify the export step: no change to
the model, a Python-free artifact, and a faster runtime.

> **Common pitfall.** Verifying an export on the example input only. Always test at least one other
> batch size and, for models with any data-dependent behaviour, inputs that exercise every branch;
> and compare *probabilities or logits*, not just the argmax, since an argmax can agree while the model
> is numerically broken.

## 4. Quantization

### 4.1 Affine quantization, derived

Quantization stores and computes with integers instead of floats. The mapping from a real value $x$ to
an integer $q$ with $b$ bits is **affine**:

$$
q = \operatorname{clamp}\!\left(\operatorname{round}\!\left(\frac{x}{s}\right) + z,\; q_{\min},\; q_{\max}\right),
\qquad
\hat{x} = s\,(q - z),
$$

where the **scale** $s > 0$ is the real-valued step between adjacent integers and the **zero-point**
$z$ is the integer that represents $x = 0$ exactly (so that zero padding and ReLU outputs are
represented without error). For a tensor whose values lie in $[x_{\min}, x_{\max}]$ there are two
standard choices:

- **Asymmetric** (unsigned, $q \in [0, 2^b - 1]$): $s = (x_{\max} - x_{\min}) / (2^b - 1)$ and
  $z = -\operatorname{round}(x_{\min} / s)$, after extending the range to include 0. It uses all
  levels for one-sided distributions such as ReLU activations.
- **Symmetric** (signed, $q \in [-2^{b-1}, 2^{b-1} - 1]$): $s = \max(|x_{\min}|, |x_{\max}|) / (2^{b-1} - 1)$
  and $z = 0$. It wastes levels for one-sided data but makes the integer arithmetic simpler, and it is
  the usual choice for weights, which are roughly symmetric around zero.

The rounding error is uniform in $[-s/2, s/2]$, so its mean square is $s^2 / 12$: halving the range
(or using one more bit) halves $s$ and therefore halves the root-mean-square error (the mean square
error drops by four). This is why **per-channel** quantization, with one scale per
output channel of a weight tensor, is more accurate than **per-tensor**: a channel with small weights
no longer has to share the scale of a channel with large ones. The payoff of the affine form is that a
matrix product can be computed entirely in integers,

$$
y = x^\top w \approx s_x s_w \,(q_x - z_x)^\top (q_w - z_w),
$$

with int8 multiplications accumulated in int32 and a single float rescale at the end; int8 matmuls run
2–4× faster than float32 on CPUs with VNNI/AMX instructions and on GPU tensor cores, and the weights
take a quarter of the memory and bandwidth. We implement the mapping and measure the error on the
trained model's largest weight matrix.

In [ ]:
def quantize_affine(x, bits=8, symmetric=True, per_channel_dim=None):
    """Affine quantization of x; returns (q, scale, zero_point) with q integer-valued (stored as float)."""
    if symmetric:
        qmax = 2 ** (bits - 1) - 1
        qmin = -qmax - 1
    else:
        qmin, qmax = 0, 2 ** bits - 1
    if per_channel_dim is None:
        x_min, x_max = x.min(), x.max()
    else:
        dims = tuple(d for d in range(x.dim()) if d != per_channel_dim)
        x_min, x_max = x.amin(dim=dims, keepdim=True), x.amax(dim=dims, keepdim=True)
    if symmetric:
        scale = torch.maximum(x_max.abs(), x_min.abs()) / qmax
        zero_point = torch.zeros_like(scale)
    else:
        x_min = torch.minimum(x_min, torch.zeros_like(x_min))       # the range must contain 0
        x_max = torch.maximum(x_max, torch.zeros_like(x_max))
        scale = (x_max - x_min) / (qmax - qmin)
        zero_point = qmin - torch.round(x_min / torch.clamp(scale, min=1e-12))
    scale = torch.clamp(scale, min=1e-12)
    q = torch.clamp(torch.round(x / scale) + zero_point, qmin, qmax)
    return q, scale, zero_point


def dequantize_affine(q, scale, zero_point):
    return (q - zero_point) * scale


w = model.fc1.weight.detach().cpu()
print(f"fc1.weight: shape {tuple(w.shape)}, range [{w.min():.3f}, {w.max():.3f}]\n")
print(f"{'bits':>4s} {'scheme':24s} {'relative error':>15s}")
schemes = [("per-tensor symmetric", dict(symmetric=True)), ("per-tensor asymmetric", dict(symmetric=False)),
           ("per-channel symmetric", dict(symmetric=True, per_channel_dim=0))]
for bits in (8, 4, 3):
    for label, kwargs in schemes:
        q, s, z = quantize_affine(w, bits, **kwargs)
        w_hat = dequantize_affine(q, s, z)
        print(f"{bits:4d} {label:24s} {((w_hat - w).norm() / w.norm()).item():15.4f}")

At 8 bits the relative error is around one percent or below for every scheme, which is why int8
post-training quantization is nearly free for most models. At 4 bits the per-tensor error rises to
well over ten percent (each halving of the bit width doubles $s$, and the extreme weights that set the
range are several standard deviations away from the bulk) and per-channel scaling helps noticeably;
at 3 bits everything degrades quickly. The plot shows the
weight distribution with the 4-bit per-tensor levels superimposed: the levels are evenly spaced, but
the weights are concentrated near zero, so most of them land on the few central levels while the
outer levels serve a handful of outliers. Non-uniform codebooks (NF4 in QLoRA, k-means quantization)
and outlier-aware schemes exist precisely to fix this mismatch at low bit widths.

In [ ]:
q4, s4, z4 = quantize_affine(w, 4, symmetric=True)
levels = dequantize_affine(torch.arange(-8, 8, dtype=torch.float32), s4, z4)
plt.figure()
plt.hist(w.flatten().cpu().numpy(), bins=120, density=True, alpha=0.7, label="fc1 weights")
for i, lv in enumerate(levels.tolist()):
    plt.axvline(lv, color="C3", lw=0.8, alpha=0.8, label="4-bit levels" if i == 0 else None)
plt.xlabel("weight value"); plt.ylabel("density"); plt.title("Weight distribution and 4-bit symmetric quantization levels")
plt.legend(); plt.tight_layout(); plt.show()

# integer matmul: quantize the activations feeding fc1 and the weights, multiply integers, rescale
with torch.inference_mode():
    mc = copy.deepcopy(model).cpu().eval()
    xb = X_test[:256].cpu()
    feats = F.max_pool2d(F.relu(mc.conv2(F.max_pool2d(F.relu(mc.conv1(xb)), 2))), 2).flatten(1)
    y_float = feats @ w.T
    qx, sx, zx = quantize_affine(feats, 8, symmetric=False)                  # activations: unsigned, per-tensor
    qw, sw, zw = quantize_affine(w, 8, symmetric=True, per_channel_dim=0)   # weights: signed, per-channel
    y_int = ((qx - zx) @ (qw - zw).T) * (sx * sw.T)                         # int32-style accumulation, one rescale
    print(f"integer matmul vs float32 output: relative error {((y_int - y_float).norm() / y_float.norm()).item():.4f}")

The integer product reproduces the float output to within a fraction of a percent. Note the two
different roles: the weight scales are known in advance (**static**), whereas the activation scale
depends on the input, which leads to the three flavours of quantization in practice:

- **Dynamic quantization** quantizes the weights offline and the activations *on the fly*, computing
  $s_x$ from each incoming batch. No calibration data is needed; the runtime cost of computing the
  activation range is small for linear layers with large weights (LSTMs, Transformers, MLP heads) and
  not worth it for convolutions, which is why PyTorch's `quantize_dynamic` targets `nn.Linear` (and
  `nn.LSTM`) by default.
- **Static post-training quantization (PTQ)** runs a few hundred calibration batches through the
  float model, records activation ranges with observers, and fixes $s_x, z_x$ per layer. The whole
  network then runs in int8 with no float conversions between layers, which is the fastest option and
  what mobile and edge runtimes use.
- **Quantization-aware training (QAT)** inserts *fake quantization* (quantize followed by dequantize)
  into the forward pass during training so that the weights adapt to the rounding; the gradient passes
  through the non-differentiable rounding with the **straight-through estimator**. It recovers most of
  the accuracy lost by PTQ at 4 bits and below.

### 4.2 Dynamic int8 quantization with `torch.ao.quantization`

`torch.ao.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)` replaces every
`nn.Linear` by a `DynamicQuantizedLinear` whose weight is stored as a packed int8 tensor with
per-tensor scale and zero-point. We measure the three quantities that matter: file size, accuracy, and
latency at batch sizes 1 and 64 on the CPU (quantized kernels are CPU kernels; the backend engine is
`fbgemm` on x86 and `qnnpack` on ARM).

In [ ]:
def state_dict_bytes(m):
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes


DYNQ_OK = False
model_cpu = copy.deepcopy(model).cpu().eval()
try:
    import torch.ao.quantization as tq
    print(f"quantized engine: {torch.backends.quantized.engine} (supported: {torch.backends.quantized.supported_engines})")
    qmodel = tq.quantize_dynamic(copy.deepcopy(model_cpu), {nn.Linear}, dtype=torch.qint8)
    print(qmodel.fc1)
    size_fp32, size_int8 = state_dict_bytes(model_cpu), state_dict_bytes(qmodel)
    acc_fp32 = evaluate(model_cpu, X_test.cpu(), y_test.cpu())[1]
    acc_int8 = evaluate(qmodel, X_test.cpu(), y_test.cpu())[1]
    print(f"\nsize: fp32 {size_fp32 / 1024:.1f} KB -> int8 dynamic {size_int8 / 1024:.1f} KB ({size_fp32 / size_int8:.2f}x smaller)")
    print(f"accuracy: fp32 {acc_fp32:.4f} -> int8 dynamic {acc_int8:.4f}")
    dyn_lat = {}
    with torch.inference_mode():
        for bs in (1, 64):
            xb = X_test[:bs].cpu().contiguous()
            for name, m in [("fp32", model_cpu), ("int8", qmodel)]:
                dyn_lat[(name, bs)] = benchmark_latency(lambda: m(xb), n_warmup=5, n_iters=30)["p50"]
            print(f"latency batch {bs:3d}: fp32 {dyn_lat[('fp32', bs)]:7.3f} ms | int8 {dyn_lat[('int8', bs)]:7.3f} ms "
                  f"| speed-up {dyn_lat[('fp32', bs)] / dyn_lat[('int8', bs)]:.2f}x")
    DYNQ_OK = True
except Exception as e:
    print("dynamic quantization is not supported on this platform:", type(e).__name__, str(e)[:200])

The linear layers hold about 98 % of this model's parameters (the `fc1` matrix alone is 1568 × 128,
some 200k of the 207k weights), so the file shrinks by close to the ideal factor of four while the
accuracy is unchanged to within one or two
test images. The latency, however, barely moves, and may even get slightly worse at batch 1: the
convolutions, which dominate the compute at 28 × 28 resolution, are still float32, and the dynamic
quantization of the activations costs a min/max pass and a conversion. This is the general lesson:
**size follows the parameters, speed follows the FLOPs**, and they live in different layers. For an
MLP or a Transformer, where linear layers dominate both, dynamic int8 typically gives a 1.5–3×
CPU speed-up (Exercise 2 quantizes the distilled MLP student to see this).

### 4.3 Fake quantization and quantization-aware training by hand

To see what QAT does we implement the two ingredients ourselves. **Fake quantization** applies
quantize-then-dequantize to a tensor inside the forward pass, so the model computes with the values it
will have after quantization while still storing float weights that the optimizer can update.
Rounding has zero gradient almost everywhere, so a plain implementation would stop learning; the
**straight-through estimator** (STE) defines the backward pass of $\hat{x} = \text{fakequant}(x)$ as
the identity, $\partial \hat{x} / \partial x := 1$. In PyTorch this is one line:
`x + (x_hat - x).detach()` evaluates to `x_hat` in the forward pass and to `x` for autograd.

The experiment trains a small MLP in float32, quantizes its weights post hoc to 8, 4 and 3 bits, and
then fine-tunes with fake-quantized 3-bit weights for one epoch. To make the comparison fair, the PTQ baseline is
given the same extra epoch of float training before being quantized.

In [ ]:
def fake_quantize(x, bits=4):
    """Quantize-dequantize with a straight-through estimator: forward uses x_hat, backward is the identity."""
    q, s, z = quantize_affine(x.detach(), bits, symmetric=True)
    x_hat = dequantize_affine(q, s, z)
    return x + (x_hat - x).detach()


class QuantMLP(nn.Module):
    """784 -> hidden -> 10 MLP whose weights can be fake-quantized in the forward pass."""

    def __init__(self, hidden=128, bits=4):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, hidden)
        self.fc2 = nn.Linear(hidden, 10)
        self.bits = bits
        self.quantize_weights = False

    def _weight(self, layer):
        return fake_quantize(layer.weight, self.bits) if self.quantize_weights else layer.weight

    def forward(self, x):
        h = F.relu(F.linear(x.flatten(1), self._weight(self.fc1), self.fc1.bias))
        return F.linear(h, self._weight(self.fc2), self.fc2.bias)


EPOCHS_Q = 1 if FAST_RUN else 3


def train_mlp(mlp, epochs, lr):
    opt = torch.optim.Adam(mlp.parameters(), lr=lr)
    for _ in range(epochs):
        train_one_epoch(mlp, opt, X_train, y_train, batch_size=64)
    return mlp


torch.manual_seed(SEED)
mlp_fp32 = train_mlp(QuantMLP().to(device), EPOCHS_Q, 1e-3)
print(f"float32 MLP after {EPOCHS_Q} epoch(s): test acc {evaluate(mlp_fp32, X_test, y_test)[1]:.4f}")

# post-training quantization baseline: one more float epoch, then quantize the weights
ptq = train_mlp(copy.deepcopy(mlp_fp32), 1, 3e-4)
ptq.quantize_weights = True
for bits in (8, 4, 3):
    ptq.bits = bits
    print(f"  PTQ {bits}-bit weights: test acc {evaluate(ptq, X_test, y_test)[1]:.4f}")

# quantization-aware training: one epoch with fake-quantized weights at 3 bits
qat = copy.deepcopy(mlp_fp32)
qat.quantize_weights, qat.bits = True, 3
qat = train_mlp(qat, 1, 3e-4)
print(f"  QAT 3-bit weights: test acc {evaluate(qat, X_test, y_test)[1]:.4f}")

Post-training quantization is essentially free at 8 bits, loses a little at 4 and a lot at 3 bits;
one epoch of quantization-aware training at 3 bits recovers most of the loss, because the weights move
to positions where the rounding does less harm. Production QAT (`torch.ao.quantization` with
`FakeQuantize` observers, or the `torchao` library) does the same for activations as well, learns the
scales, and fuses BatchNorm into the preceding convolution before quantizing so that the fused weights
are what gets rounded.

### 4.4 Quantization inside ONNX Runtime

Quantization can also be applied to the exported graph, which is convenient because the same tool
serves any framework. ONNX Runtime's `quantize_dynamic` rewrites `MatMul` nodes into `MatMulInteger`
with a `DynamicQuantizeLinear` in front of them (by default it also rewrites `Conv` into `ConvInteger`,
whose CPU kernel exists only for unsigned weights; we restrict it to `MatMul` with
`op_types_to_quantize`, so that the comparison with PyTorch's dynamic quantization of the linear
layers is like for like). One wrinkle: PyTorch exports a 2-D linear layer as a fused `Gemm` node,
which the dynamic quantizer leaves alone; a small graph rewrite, splitting each `Gemm` into
`MatMul` + `Add`, exposes the matmul to the quantizer. The rewrite is a useful illustration of ONNX
graph surgery with the `onnx.helper` API, and the cell reports which operators ended up quantized,
together with the size, accuracy and latency of the result.

In [ ]:
def gemm_to_matmul(onnx_model):
    """Rewrite each Gemm(A, B, C) with alpha = beta = 1 as MatMul(A, B') + Add(C); returns the number rewritten."""
    from onnx import helper, numpy_helper
    g = onnx_model.graph
    inits = {t.name: t for t in g.initializer}
    new_nodes, n_rewritten = [], 0

    def keep(node):                                                 # detached copy of an existing node
        n = onnx.NodeProto()
        n.CopyFrom(node)
        new_nodes.append(n)

    for node in g.node:
        if node.op_type != "Gemm" or node.input[1] not in inits:
            keep(node)
            continue
        attrs = {a.name: helper.get_attribute_value(a) for a in node.attribute}
        if attrs.get("alpha", 1.0) != 1.0 or attrs.get("beta", 1.0) != 1.0 or attrs.get("transA", 0) != 0:
            keep(node)
            continue
        W = numpy_helper.to_array(inits[node.input[1]])
        if attrs.get("transB", 0) == 1:
            W = W.T
        w_name = node.input[1] + "_matmul"
        g.initializer.append(numpy_helper.from_array(np.ascontiguousarray(W), w_name))
        has_bias = len(node.input) > 2 and node.input[2] != ""
        mm_out = node.output[0] + "_matmul" if has_bias else node.output[0]
        new_nodes.append(helper.make_node("MatMul", [node.input[0], w_name], [mm_out], name=(node.name or "gemm") + "_matmul"))
        if has_bias:
            new_nodes.append(helper.make_node("Add", [mm_out, node.input[2]], [node.output[0]], name=(node.name or "gemm") + "_add"))
        n_rewritten += 1
    g.ClearField("node")
    g.node.extend(new_nodes)
    used = {name for n in g.node for name in n.input}
    for t in list(g.initializer):                                   # drop the now-unused Gemm weights
        if t.name not in used:
            g.initializer.remove(t)
    return n_rewritten


if HAVE_ONNX and ORT_OK:
    try:
        from onnxruntime.quantization import quantize_dynamic, QuantType
        mm_path, q_path = MODEL_DIR / "mnist_cnn_matmul.onnx", MODEL_DIR / "mnist_cnn_int8.onnx"
        m_mm = onnx.load(str(onnx_path))
        n_rw = gemm_to_matmul(m_mm)
        onnx.checker.check_model(m_mm)
        onnx.save(m_mm, str(mm_path))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            quantize_dynamic(str(mm_path), str(q_path), weight_type=QuantType.QInt8, op_types_to_quantize=["MatMul"])
        m_q = onnx.load(str(q_path))
        print(f"rewrote {n_rw} Gemm node(s); quantized graph operators: {dict(Counter(n.op_type for n in m_q.graph.node))}")
        sess_q = ort.InferenceSession(str(q_path), providers=["CPUExecutionProvider"])
        acc_q = ort_accuracy(sess_q, X_test_raw, y_test)
        print(f"size: {onnx_path.stat().st_size / 1024:.1f} KB -> {q_path.stat().st_size / 1024:.1f} KB | "
              f"accuracy: fp32 ORT {acc_ort:.4f} -> int8 ORT {acc_q:.4f}")
        for bs in (1, 64):
            xb_np = X_test_raw[:bs].cpu().numpy().astype(np.float32)
            st_q = benchmark_latency(lambda: sess_q.run(None, {in_name: xb_np}), n_warmup=5, n_iters=30)
            print(f"latency batch {bs:3d}: fp32 ORT {ort_lat[bs]:7.3f} ms | int8 ORT {st_q['p50']:7.3f} ms | speed-up {ort_lat[bs] / st_q['p50']:.2f}x")
    except Exception as e:
        print("ONNX Runtime quantization failed here:", type(e).__name__, str(e)[:200])
else:
    print("ONNX Runtime quantization skipped (onnx/onnxruntime unavailable or export failed).")

As with PyTorch's dynamic quantization, the file shrinks substantially while the accuracy stays put,
and the latency gain is modest because the convolutions remain in float. For static int8 including
convolutions, ONNX Runtime offers `quantize_static` with a calibration data reader, and TensorRT and
OpenVINO perform the same calibration on their side; that is where the multi-fold speed-ups reported
for int8 inference come from.

> **Key idea.** Quantization error is $s^2/12$ per element: use per-channel scales and 8 bits for a
> free win, and reach for QAT (fake quantization with the straight-through estimator) at 4 bits and
> below. Size savings follow the parameters, speed savings follow the FLOPs.

## 5. Pruning

### 5.1 Magnitude pruning, unstructured and structured

Trained networks are redundant: many weights are close to zero and contribute little. **Pruning**
removes them. The simplest and still most competitive criterion is magnitude: rank the weights by
$|w|$ and zero out the smallest fraction $s$ (the **sparsity**). Two variants matter:

- **Unstructured** pruning zeroes individual weights anywhere in a tensor. It reaches high sparsity
  (80–95 % on over-parameterized models) with little accuracy loss, especially with a short
  fine-tuning after pruning, and iterating prune-then-fine-tune reaches higher sparsity than pruning
  in one shot (Han et al., 2015). Its weakness is that a dense matmul does not care that 90 % of the
  entries are zero: it multiplies them anyway. Unstructured sparsity therefore reduces the *size* of a
  model (in a sparse storage format) but not its *latency* on ordinary CPUs and GPUs, unless the
  sparsity is very high and a sparse kernel is used, or the hardware supports a specific pattern
  (NVIDIA's 2:4 semi-structured sparsity, in which two of every four consecutive weights are zero,
  gives a genuine 2× on Ampere tensor cores).
- **Structured** pruning removes whole channels, filters, attention heads or layers. The remaining
  network is simply a smaller dense network, so it *is* faster on any hardware, but the accuracy cost
  per removed parameter is higher and the choice of which structures to remove is harder.

`torch.nn.utils.prune` implements both by *reparameterizing* a module: the original tensor is renamed
`weight_orig`, a buffer `weight_mask` is added, and a forward pre-hook recomputes
`weight = weight_orig * weight_mask` before every call. Gradients still flow to `weight_orig`, but
through the mask, so pruned entries receive zero gradient and stay zero during fine-tuning.
`prune.remove` makes the pruning permanent by writing the masked tensor back as `weight`.
`prune.global_unstructured` ranks weights across several layers jointly, which is better than
pruning every layer by the same fraction because layers differ in redundancy.

In [ ]:
import torch.nn.utils.prune as prune

PRUNE_LAYERS = ["conv1", "conv2", "fc1", "fc2"]


def prunable(m):
    return [(getattr(m, name), "weight") for name in PRUNE_LAYERS]


def prune_global_copy(base, amount):
    """Deep-copy base and zero the globally smallest `amount` fraction of its weights (L1 magnitude)."""
    m = copy.deepcopy(base)
    if amount > 0:
        prune.global_unstructured(prunable(m), pruning_method=prune.L1Unstructured, amount=amount)
    return m


def layer_sparsity(m):
    return {name: (getattr(m, name).weight == 0).float().mean().item() for name in PRUNE_LAYERS}


def total_sparsity(m):
    zeros = sum((getattr(m, name).weight == 0).sum().item() for name in PRUNE_LAYERS)
    total = sum(getattr(m, name).weight.numel() for name in PRUNE_LAYERS)
    return zeros / total


sparsities = [0.0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98]
prune_accs = []
for s in sparsities:
    pm = prune_global_copy(model, s)
    acc = evaluate(pm, X_test, y_test)[1]
    prune_accs.append(acc)
    print(f"global sparsity {s:4.0%}: test acc {acc:.4f} | per layer " +
          ", ".join(f"{k} {v:.0%}" for k, v in layer_sparsity(pm).items()))

plt.figure()
plt.plot([s * 100 for s in sparsities], [a * 100 for a in prune_accs], "o-")
plt.axhline(BASE_ACC * 100, color="grey", ls="--", label="dense model")
plt.xlabel("global weight sparsity (%)"); plt.ylabel("test accuracy (%)")
plt.title("One-shot global magnitude pruning without fine-tuning"); plt.legend()
plt.tight_layout(); plt.show()

The curve is flat up to about 70–80 % sparsity and then falls off a cliff: the model can lose most of
its weights without retraining. The per-layer numbers show what global pruning does: the large
`fc1` matrix, which holds most of the parameters and has the smallest typical weights, is pruned far
more than the convolutions, whose few weights are individually important. Beyond the cliff, a short
fine-tuning brings much of the accuracy back, because the surviving weights can re-adjust to the
missing ones; iterating this (Exercise 3) pushes the cliff further out.

In [ ]:
EPOCHS_FT = 1 if FAST_RUN else 2
SPARSITY_FT = 0.95
pm = prune_global_copy(model, SPARSITY_FT)
acc_before = evaluate(pm, X_test, y_test)[1]
opt_ft = torch.optim.Adam(pm.parameters(), lr=3e-4)
for _ in range(EPOCHS_FT):
    train_one_epoch(pm, opt_ft, X_train, y_train)
acc_after = evaluate(pm, X_test, y_test)[1]
print(f"{SPARSITY_FT:.0%} sparsity: acc {acc_before:.4f} -> {acc_after:.4f} after {EPOCHS_FT} fine-tuning epoch(s); "
      f"sparsity still {total_sparsity(pm):.3f}")
print("state_dict keys while pruned :", [k for k in pm.state_dict().keys() if "fc2" in k])
for module, name in prunable(pm):
    prune.remove(module, name)
print("state_dict keys after remove :", [k for k in pm.state_dict().keys() if "fc2" in k],
      f"| sparsity {total_sparsity(pm):.3f}")

# structured pruning: remove half of conv2's output channels by their L2 norm
ps = copy.deepcopy(model)
prune.ln_structured(ps.conv2, "weight", amount=0.5, n=2, dim=0)
zero_channels = int((ps.conv2.weight.flatten(1).abs().sum(1) == 0).sum().item())
print(f"\nstructured: {zero_channels}/{ps.conv2.out_channels} conv2 channels zeroed -> test acc {evaluate(ps, X_test, y_test)[1]:.4f} (no fine-tuning)")

Fine-tuning recovers most of the lost accuracy even at 95 % sparsity, the masks survive training, and
`prune.remove` folds them into an ordinary `weight` tensor. Structured pruning of half of `conv2`'s
channels costs more accuracy per parameter, but the pruned channels could be physically deleted (along
with the matching input channels of `fc1`) to obtain a smaller, genuinely faster dense model, which is
something no unstructured mask can offer. The final cell of this section verifies the claim about
latency directly.

In [ ]:
dense_cpu = copy.deepcopy(model).cpu().eval()
sparse_cpu = copy.deepcopy(pm).cpu().eval()                       # 95 % of weights are exact zeros
xb_cpu = X_test[:64].cpu().contiguous()
with torch.inference_mode():
    st_dense = benchmark_latency(lambda: dense_cpu(xb_cpu), n_warmup=5, n_iters=30)
    st_sparse = benchmark_latency(lambda: sparse_cpu(xb_cpu), n_warmup=5, n_iters=30)
print(f"batch 64 CPU latency: dense {st_dense['p50']:.3f} ms | 95%-sparse weights in dense format {st_sparse['p50']:.3f} ms")

try:
    W = sparse_cpu.fc1.weight.detach()
    x_feat = torch.randn(64, W.shape[1])
    W_sparse = W.to_sparse_csr()
    with torch.inference_mode():
        st_mm_dense = benchmark_latency(lambda: x_feat @ W.T, n_warmup=5, n_iters=50)
        st_mm_sparse = benchmark_latency(lambda: (W_sparse @ x_feat.T).T, n_warmup=5, n_iters=50)
    print(f"fc1 matmul alone: dense {st_mm_dense['p50']:.4f} ms | CSR sparse ({1 - W_sparse.values().numel() / W.numel():.0%} zeros) "
          f"{st_mm_sparse['p50']:.4f} ms")
except Exception as e:
    print("sparse matmul not available here:", type(e).__name__)

The pruned model in dense format runs in exactly the same time as the dense one: the hardware
multiplies zeros as fast as any other number. A sparse format does skip the zeros, but the
bookkeeping of indices costs more than it saves unless sparsity is extreme, and on this small matrix
the CSR product is often *slower* than the dense one. This is why, for latency, practitioners reach
for structured pruning, distillation into a smaller architecture (next section), or hardware-specific
patterns such as 2:4, and use unstructured pruning mainly to shrink storage and bandwidth.

## 6. Knowledge distillation

### 6.1 Soft targets and temperature

A large, accurate **teacher** model knows more than its top-1 prediction: its full output
distribution encodes which classes are similar (a 7 looks more like a 1 than like a 0). Hinton et al.
(2015) proposed to train a small **student** to match the teacher's distribution rather than only the
hard labels. Because a trained classifier's softmax is very peaked, the similarity structure is
revealed by *softening* the logits with a temperature $T$:

$$
p_i^{(T)} = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)} .
$$

At $T = 1$ this is the ordinary softmax; as $T$ grows the distribution flattens and the small
probabilities of the wrong classes become visible. The distillation loss combines a soft term, the
KL divergence between the softened teacher and student distributions, with the usual cross-entropy
on the true labels:

$$
\mathcal{L} = \alpha\, T^2\, \mathrm{KL}\!\left(p_{\text{teacher}}^{(T)} \,\|\, p_{\text{student}}^{(T)}\right)
+ (1 - \alpha)\, \mathrm{CE}\!\left(y, p_{\text{student}}^{(1)}\right).
$$

The factor $T^2$ compensates for the fact that the gradients of the soft term scale as $1/T^2$: the
gradient with respect to a student logit is $\frac{1}{T}\,(p^{(T)}_{\text{student},i} - p^{(T)}_{\text{teacher},i})$,
one factor $1/T$ coming from the chain rule through $z_i / T$, and the difference of the two softened
distributions itself shrinks like $1/T$ for large $T$. Multiplying by $T^2$ keeps the relative weight
of the two terms roughly constant when $T$ is tuned. Why does matching soft targets help? Each
training example now carries ten informative numbers instead of one, the targets are smoother (which
regularizes, much as label smoothing does), and the teacher's mistakes on ambiguous examples tell the
student how ambiguous they are. The effect is largest when the student is much smaller than the
teacher and the data set is small.

We first look at the teacher's softened distribution for a test digit, then distil the CNN into a
small MLP and compare with the same MLP trained from scratch on the same batches.

In [ ]:
teacher = model.eval()
with torch.inference_mode():
    teacher_logits = torch.cat([teacher(X_train[s:s + 500]) for s in range(0, N_TRAIN, 500)])
    z = teacher(X_test[:1])[0]
teacher_logits = teacher_logits.clone()          # a normal tensor: inference tensors cannot take part in autograd
print("teacher logits on the training set:", tuple(teacher_logits.shape))

temps = [1, 2, 5, 10]
fig, axes = plt.subplots(1, len(temps) + 1, figsize=(12, 2.8))
axes[0].imshow(X_test_raw[0, 0].cpu(), cmap="gray"); axes[0].set_title(f"label {y_test[0].item()}"); axes[0].axis("off")
for ax, T in zip(axes[1:], temps):
    p = np.asarray(F.softmax(z / T, dim=0).cpu())
    ax.bar(range(10), p, color="C0"); ax.set_ylim(0, 1); ax.set_xticks(range(10))
    ax.set_title(f"T = {T}"); ax.grid(False)
plt.suptitle("Teacher probabilities at increasing temperature"); plt.tight_layout(); plt.show()

At $T = 1$ the teacher is almost certain; at $T = 5$ or $10$ the runner-up classes (the ones that
share strokes with the true digit) become visible, and that is the information the student is meant to
absorb. In the distillation loss we use $T = 4$ and $\alpha = 0.7$, common defaults for small
classification problems.

In [ ]:
def distillation_loss(student_logits, teacher_logits, y, T=4.0, alpha=0.7):
    """alpha * T^2 * KL(teacher_T || student_T) + (1 - alpha) * CE(y, student)."""
    soft = F.kl_div(F.log_softmax(student_logits / T, dim=1), F.softmax(teacher_logits / T, dim=1),
                    reduction="batchmean") * (T * T)
    hard = F.cross_entropy(student_logits, y)
    return alpha * soft + (1 - alpha) * hard


class StudentMLP(nn.Module):
    """A 784 -> 64 -> 10 MLP: ~50k parameters, about a quarter of the CNN and no convolutions."""

    def __init__(self, hidden=64):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, hidden)
        self.fc2 = nn.Linear(hidden, 10)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x.flatten(1))))


def train_student(distill, epochs, T=4.0, alpha=0.7, seed=SEED, hidden=64):
    torch.manual_seed(seed)
    student = StudentMLP(hidden).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=1e-3)
    loss_fn = (lambda logits, yb, idx: distillation_loss(logits, teacher_logits[idx], yb, T, alpha)) if distill else None
    history = []
    for ep in range(epochs):
        torch.manual_seed(seed + 1 + ep)                     # identical mini-batch order for both students
        train_one_epoch(student, opt, X_train, y_train, 64, loss_fn)
        history.append(evaluate(student, X_test, y_test)[1])
    return student, history


EPOCHS_KD = 3 if FAST_RUN else 8
t0 = time.perf_counter()
student_scratch, hist_scratch = train_student(distill=False, epochs=EPOCHS_KD)
student_kd, hist_kd = train_student(distill=True, epochs=EPOCHS_KD)
print(f"student params {count_params(student_kd):,} vs teacher {count_params(teacher):,} | {time.perf_counter() - t0:.1f}s")
print(f"teacher acc {BASE_ACC:.4f} | student from scratch {hist_scratch[-1]:.4f} | distilled student {hist_kd[-1]:.4f}")

plt.figure()
plt.plot(range(1, EPOCHS_KD + 1), [a * 100 for a in hist_scratch], "o-", label="student, hard labels")
plt.plot(range(1, EPOCHS_KD + 1), [a * 100 for a in hist_kd], "s-", label="student, distilled (T=4)")
plt.axhline(BASE_ACC * 100, color="grey", ls="--", label="teacher CNN")
plt.xlabel("epoch"); plt.ylabel("test accuracy (%)"); plt.title("Knowledge distillation: CNN teacher to MLP student")
plt.legend(); plt.tight_layout(); plt.show()

The distilled student is consistently ahead of the student trained on hard labels, typically by
0.3–1 percentage point on this task, and the gap opens early: the soft targets act as a richer, less
noisy training signal. Neither student reaches the teacher, which is the point: the MLP has no
convolutions and a quarter of the parameters, and Exercise 2 shows that it is also the model that
benefits most from int8 quantization. Distillation is the workhorse behind most "small but strong"
models (DistilBERT, TinyBERT, and many mobile vision models), and its modern variants distil
intermediate representations and attention maps as well as outputs.

> **Key idea.** Compression is a menu: quantization shrinks bytes per weight, pruning shrinks the
> number of weights (but rarely the time), and distillation shrinks the *architecture*, which is the
> only one of the three that reliably shrinks latency on any hardware.

## 7. Serving

### 7.1 An inference service with FastAPI and ONNX Runtime

A model becomes a product through an API. The minimal contract is an HTTP endpoint that accepts a
batch of inputs in a validated schema, returns predictions with the model version, and exposes a
health check that the orchestrator can poll. The service below is written to disk rather than
executed (a notebook is not a web server, and `fastapi` is not part of the course environment), and
it is complete: copy the folder next to the ONNX model, install `fastapi`, `uvicorn` and
`onnxruntime`, and run `uvicorn app:app`. Its design decisions are worth reading line by line:

- the model is loaded **once**, in the application's *lifespan* handler, not per request;
- requests and responses are **pydantic** models, so malformed input is rejected with a 422 before
  it reaches the model, and the schema is published automatically at `/docs`;
- the endpoint accepts a **list of images** so that a client can batch, and rejects oversized batches
  (a form of load shedding);
- the response carries the **model version** and the server-side latency, which the monitoring of
  Section 8 relies on;
- ONNX Runtime is configured with a small thread count per worker, and the process count is set at
  the `uvicorn` level: for CPU inference, several single-threaded workers beat one multi-threaded one.

In [ ]:
SERVING_DIR = DATA_DIR / "serving_nb15"
SERVING_DIR.mkdir(parents=True, exist_ok=True)

APP_PY = r'''
"""FastAPI inference service for the MNIST classifier (ONNX Runtime backend).

Run locally (fastapi and uvicorn are not part of the course environment):
    python -m pip install -r requirements.txt
    MODEL_PATH=../models_nb15/mnist_cnn.onnx uvicorn app:app --host 0.0.0.0 --port 8000 --workers 2
Interactive docs: http://localhost:8000/docs
"""
import os
import time
from contextlib import asynccontextmanager
from typing import List

import numpy as np
import onnxruntime as ort
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

MODEL_PATH = os.environ.get("MODEL_PATH", "mnist_cnn.onnx")
MODEL_VERSION = os.environ.get("MODEL_VERSION", "1.0.0")
MAX_BATCH = int(os.environ.get("MAX_BATCH", "256"))
state = {}


class PredictRequest(BaseModel):
    images: List[List[float]] = Field(..., description="each image: 784 grey values in [0, 1], row-major 28x28")


class Prediction(BaseModel):
    label: int
    probability: float
    probs: List[float]


class PredictResponse(BaseModel):
    predictions: List[Prediction]
    model_version: str
    server_latency_ms: float


@asynccontextmanager
async def lifespan(app):
    """Load the model once per worker process; release it on shutdown."""
    options = ort.SessionOptions()
    options.intra_op_num_threads = int(os.environ.get("ORT_THREADS", "1"))
    state["session"] = ort.InferenceSession(MODEL_PATH, options, providers=["CPUExecutionProvider"])
    state["input_name"] = state["session"].get_inputs()[0].name
    state["n_images"] = 0
    yield
    state.clear()


app = FastAPI(title="MNIST classifier", version=MODEL_VERSION, lifespan=lifespan)


@app.get("/health")
def health():
    return {"status": "ok", "model_version": MODEL_VERSION, "images_served": state.get("n_images", 0)}


@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    t0 = time.perf_counter()
    if not 0 < len(req.images) <= MAX_BATCH:
        raise HTTPException(status_code=413, detail=f"batch size must be between 1 and {MAX_BATCH}")
    try:
        x = np.asarray(req.images, dtype=np.float32).reshape(-1, 1, 28, 28)
    except ValueError:
        raise HTTPException(status_code=422, detail="every image must contain exactly 784 values")
    if x.min() < 0.0 or x.max() > 1.0:
        raise HTTPException(status_code=422, detail="pixel values must lie in [0, 1]")
    probs = state["session"].run(None, {state["input_name"]: x})[0]
    state["n_images"] += x.shape[0]
    predictions = [Prediction(label=int(p.argmax()), probability=float(p.max()), probs=[float(v) for v in p]) for p in probs]
    return PredictResponse(predictions=predictions, model_version=MODEL_VERSION,
                           server_latency_ms=(time.perf_counter() - t0) * 1000.0)
'''.lstrip()

CLIENT_PY = r'''
"""Minimal client for the MNIST service: sends a batch of images and prints the predictions."""
import sys
import time

import numpy as np
import requests

URL = sys.argv[1] if len(sys.argv) > 1 else "http://localhost:8000"
images = np.random.default_rng(0).random((4, 784)).round(2).tolist()    # replace with real 28x28 images in [0, 1]

print(requests.get(f"{URL}/health", timeout=5).json())
t0 = time.perf_counter()
resp = requests.post(f"{URL}/predict", json={"images": images}, timeout=10)
resp.raise_for_status()
body = resp.json()
print(f"round trip {1000 * (time.perf_counter() - t0):.1f} ms, server {body['server_latency_ms']:.2f} ms, model {body['model_version']}")
for i, p in enumerate(body["predictions"]):
    print(f"image {i}: label {p['label']} (p = {p['probability']:.3f})")
'''.lstrip()

DOCKERFILE = r'''
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN python -m pip install --no-cache-dir -r requirements.txt
COPY app.py mnist_cnn.onnx ./
ENV MODEL_PATH=/app/mnist_cnn.onnx MODEL_VERSION=1.0.0 ORT_THREADS=1
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''.lstrip()

REQUIREMENTS = "fastapi>=0.110\nuvicorn[standard]>=0.29\nonnxruntime>=1.17\nnumpy>=1.26\npydantic>=2.0\n"

for fname, content in [("app.py", APP_PY), ("client.py", CLIENT_PY), ("Dockerfile", DOCKERFILE), ("requirements.txt", REQUIREMENTS)]:
    (SERVING_DIR / fname).write_text(content, encoding="utf-8")
    if fname.endswith(".py"):
        compile(content, fname, "exec")                     # syntax check only; fastapi is not imported here
    print(f"wrote {SERVING_DIR / fname} ({len(content.splitlines())} lines)")
if ONNX_OK:
    (SERVING_DIR / "mnist_cnn.onnx").write_bytes(onnx_path.read_bytes())
    print("copied mnist_cnn.onnx next to the service")
print("\n" + APP_PY)

Reading the service: `lifespan` runs once per worker process and builds the ONNX Runtime session;
`/health` answers instantly and reports the version and a counter, which is what Kubernetes liveness
and readiness probes call; `/predict` validates, reshapes, checks the value range (a cheap guard
against clients sending 0–255 bytes instead of $[0, 1]$ floats, the single most common integration
bug for this kind of model), runs the session and returns typed predictions. The client sends a JSON
batch and prints server-side and round-trip latency, whose difference is the network and
serialization cost, which for small models often exceeds the inference itself; binary encodings
(protobuf via gRPC, or NumPy bytes) are the remedy when it matters. The Dockerfile packages the
service with pinned dependencies and a health check; `--workers 2` starts two processes, each with a
single-threaded ONNX Runtime session, which is the standard CPU serving layout.

### 7.2 Batching, GPUs and the serving landscape

**Dynamic batching.** A single request uses a fraction of the hardware (Section 1.3). Production
servers therefore queue incoming requests for up to a few milliseconds (`max_wait`) or until a batch
of `max_batch` is collected, run them together, and split the results. The queue adds latency equal
to the wait plus the batch's compute time, but multiplies throughput; the right settings are found
from the latency/throughput curve and the service-level objective (Exercise 5 simulates this
trade-off). In the FastAPI service above, batching is left to the client; a dynamic batcher would be
an `asyncio` queue with a background task that drains it.

**GPU serving** changes the arithmetic: the knee of the throughput curve sits at hundreds of images,
so batching is mandatory, requests should be pinned and transferred asynchronously, and the model
should run in float16/bfloat16 with `channels_last` (Notebook 14 · Efficient Training and Scaling) or
through TensorRT. A GPU worker
is also expensive to keep idle, which is why GPU services are usually shared by several models.

The dedicated serving systems automate all of this:

- **TorchServe** (PyTorch's own server, in maintenance) wraps a model archive with a handler class,
  provides dynamic batching, versioning and metrics endpoints.
- **NVIDIA Triton Inference Server** serves ONNX, TensorRT, PyTorch and other backends side by side,
  with dynamic batching, model ensembles, concurrent model execution on one GPU, and Prometheus
  metrics; it is the standard choice for GPU fleets.
- **vLLM**, **TensorRT-LLM** and **SGLang** serve large language models, where the batching problem
  is different: sequences finish at different times, so they use *continuous batching* (new requests
  join the batch at every decoding step) and *paged attention* to manage the KV cache (Notebook 09 ·
  Attention and Transformers) as virtual memory.
- **Edge runtimes**: **ExecuTorch** takes a `torch.export` program to Android, iOS and
  microcontrollers; **TensorRT** compiles for NVIDIA GPUs including Jetson; **Core ML** targets
  Apple devices (via `coremltools` from a traced or exported model); **ONNX Runtime Mobile/Web**
  runs ONNX on phones and in browsers via WebAssembly and WebGPU; **LiteRT** (formerly TensorFlow
  Lite) is the counterpart in the TensorFlow ecosystem. All of these prefer static shapes and
  int8 or float16 models, which is why the export and quantization work of Sections 3 and 4 is the
  prerequisite for edge deployment.

## 8. Operating models: monitoring, drift, rollback and model cards

### 8.1 Monitoring inputs and outputs

A deployed model has no test set. Labels for production traffic arrive late (when a user complains,
when a fraud case is resolved) or never, so the accuracy of the model in production is unknown at the
moment it matters. What *can* be observed immediately are the inputs and the outputs, and their
distributions are the early-warning system. Three kinds of change are distinguished:

- **Data drift** (covariate shift): the input distribution $p(x)$ changes, for instance a new camera
  with different contrast, a new customer segment, a new season.
- **Concept drift**: the relation $p(y \mid x)$ changes; the same input should now get a different
  answer (fraud patterns evolve, product categories are redefined).
- **Prediction drift**: the output distribution changes, which is the observable consequence of
  either of the above or of a pipeline bug (a client suddenly sends unnormalized bytes).

A standard statistic for comparing a production window with a reference window (the validation set at
training time, or the first week of production) is the **Population Stability Index**,

$$
\mathrm{PSI} = \sum_{k=1}^{K} (q_k - p_k) \ln \frac{q_k}{p_k},
$$

where $p_k$ and $q_k$ are the fractions of reference and production samples falling into bin $k$ of a
histogram (the bins are usually the deciles of the reference). PSI is a symmetrized KL divergence
between the two histograms; the folklore thresholds are below 0.1 "stable", 0.1–0.25 "moderate shift,
investigate", above 0.25 "significant shift, act". We compute it for two input statistics (mean
intensity, fraction of bright pixels) and two output statistics (confidence, predicted class) under
three scenarios: an unchanged distribution, images with lower contrast and sensor noise, and inverted
images (white digits on black becoming black on white), which a real camera pipeline change could
easily produce.

In [ ]:
def psi(reference, production, bins=10, eps=1e-6):
    """Population Stability Index between two samples of a scalar statistic (bins = reference deciles)."""
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1))).astype(float)
    if len(edges) < 4:                                   # heavily tied reference values: use equal-width bins instead
        lo, hi = min(reference.min(), production.min()), max(reference.max(), production.max())
        edges = np.linspace(lo, hi + 1e-9, bins + 1)
    edges[0], edges[-1] = -np.inf, np.inf
    p = np.histogram(reference, edges)[0] / len(reference) + eps
    q = np.histogram(production, edges)[0] / len(production) + eps
    return float(np.sum((q - p) * np.log(q / p)))


def psi_categorical(reference, production, n_classes=10, eps=1e-6):
    p = np.bincount(reference, minlength=n_classes) / len(reference) + eps
    q = np.bincount(production, minlength=n_classes) / len(production) + eps
    return float(np.sum((q - p) * np.log(q / p)))


@torch.inference_mode()
def monitoring_stats(model, X_raw):
    """Input and output statistics a monitor would log for a batch of raw images."""
    model.eval()
    probs = F.softmax(model((X_raw - MNIST_MEAN) / MNIST_STD), dim=1)
    return {"mean_intensity": np.asarray(X_raw.mean(dim=(1, 2, 3)).cpu()),
            "bright_fraction": np.asarray((X_raw > 0.5).float().mean(dim=(1, 2, 3)).cpu()),
            "confidence": np.asarray(probs.max(dim=1).values.cpu()),
            "predicted_class": np.asarray(probs.argmax(dim=1).cpu())}


half = N_TEST // 2
reference_raw, reference_y = X_test_raw[:half], y_test[:half]
torch.manual_seed(SEED)
scenarios = {
    "unchanged": X_test_raw[half:],
    "low contrast + noise": (X_test_raw[half:] * 0.6 + 0.2 + 0.1 * torch.randn_like(X_test_raw[half:])).clamp(0, 1),
    "inverted": 1.0 - X_test_raw[half:],
}
ref_stats = monitoring_stats(model, reference_raw)
print(f"{'scenario':22s} {'PSI intensity':>13s} {'PSI bright':>10s} {'PSI confidence':>14s} {'PSI class':>9s} {'accuracy':>9s}")
for name, X_prod in scenarios.items():
    st = monitoring_stats(model, X_prod)
    acc = float((torch.as_tensor(st["predicted_class"]) == y_test[half:].cpu()).float().mean())
    print(f"{name:22s} {psi(ref_stats['mean_intensity'], st['mean_intensity']):13.3f} "
          f"{psi(ref_stats['bright_fraction'], st['bright_fraction']):10.3f} "
          f"{psi(ref_stats['confidence'], st['confidence']):14.3f} "
          f"{psi_categorical(ref_stats['predicted_class'], st['predicted_class']):9.3f} {acc:9.4f}")

The unchanged scenario gives PSI values near zero on every statistic, as it should. Both shifted
scenarios are flagged loudly by the input statistics *before any label is needed*, and the output
statistics tell the rest of the story: under the inversion the confidence distribution collapses and
the predicted classes pile up on a few digits, and the accuracy (which the monitor could not have
known) has fallen close to chance level. A monitor that logs a few such statistics per window, computes PSI
against a reference, and alerts above a threshold is inexpensive and catches the majority of
production incidents, most of which are pipeline changes rather than subtle statistical drift.

### 8.2 Rolling out and rolling back

Because the production accuracy of a new model cannot be measured offline, deployments are staged:

- **Shadow deployment**: the new model receives a copy of the traffic and its predictions are logged
  but not served; agreement with the old model and the latency distribution are checked for free.
- **Canary release**: a small fraction of traffic (1–5 %) is served by the new model; error rates,
  latency and business metrics are compared with the rest, and the fraction is increased in steps.
- **A/B test**: two models serve randomly assigned user groups long enough to measure a business
  metric with statistical power; the sample size follows from the effect size one wants to detect,
  and the assignment must be sticky per user to avoid inconsistent experiences.
- **Rollback**: promotion and demotion are registry operations (Section 2.2): the serving
  configuration points at a model version, and reverting it is one change with no retraining. Keeping
  the previous version warm makes the rollback take seconds.

None of this is possible without **reproducibility of the pipeline**: the training data snapshot
(versioned with a hash or a tool such as DVC), the code commit, the configuration and the environment
that produced each model version must be recorded so that a version can be rebuilt or audited. The
bundle of Section 2.2 recorded most of these; a real registry adds the data version and the commit.

### 8.3 Cost estimates

A capacity plan follows from the numbers we measured. If a service must handle $r$ requests per
second at a p99 latency budget, the batch size at which one worker meets the budget (Section 1.3) and
the resulting throughput per worker give the number of workers, and the price per worker-hour gives
the monthly cost. The cell below does this arithmetic with the measured batch-1 and batch-64 latencies
of the ONNX Runtime model if available (eager PyTorch otherwise), for a small and a large load, with
illustrative instance prices.

In [ ]:
lat_b1 = ort_lat[1] if ORT_OK else lat_rows[0][1]["p50"]
lat_b64 = ort_lat[64] if ORT_OK else lat_rows[3][1]["p50"]
throughput_per_worker = {1: 1000.0 / lat_b1, 64: 64 * 1000.0 / lat_b64}
print(f"one CPU worker: {throughput_per_worker[1]:8.0f} req/s at batch 1 ({lat_b1:.2f} ms), "
      f"{throughput_per_worker[64]:8.0f} req/s at batch 64 ({lat_b64:.2f} ms per batch)")
print(f"\n{'load (req/s)':>12s} {'batch':>5s} {'workers (with 2x headroom)':>27s} {'monthly cost at $0.05/worker-hour':>34s}")
for load in (100, 2_000, 50_000):
    for bs in (1, 64):
        workers = math.ceil(2 * load / throughput_per_worker[bs])
        print(f"{load:12,d} {bs:5d} {workers:27d} {workers * 0.05 * 24 * 30:31,.0f} USD")

The table makes the economics of batching explicit: at high load, serving at batch 64 needs far fewer
workers than serving each request alone, and the price of that saving is the few milliseconds of
queueing latency that the batcher adds. It also shows why quantization and distillation matter
commercially: a 2× faster model halves this table.

### 8.4 The model card

Finally, the model needs documentation that travels with it. **Model cards** (Mitchell et al., 2019)
are a short, standardized report of what a model is, what it is for, how it was evaluated, and where it
fails, written for the people who will use or be affected by it rather than for the people who built
it. The cell below writes a card for our classifier with the numbers measured in this notebook and a
per-class analysis, and stores it next to the model bundle. Notebook 17 · Capstone Projects reuses
this template.

In [ ]:
@torch.inference_mode()
def per_class_accuracy(model, X, y, n_classes=10):
    model.eval()
    preds = torch.cat([model(X[s:s + 500]).argmax(1) for s in range(0, X.shape[0], 500)])
    return [float((preds[y == c] == c).float().mean()) for c in range(n_classes)]


pca = per_class_accuracy(model, X_test, y_test)
worst = int(np.argmin(pca))
size_kb = bundle_path.stat().st_size / 1024
int8_note = (f"Dynamic int8 quantization of the linear layers keeps accuracy at {acc_int8:.4f} and reduces the file to "
             f"{size_int8 / 1024:.0f} KB." if DYNQ_OK else "Dynamic int8 quantization was not evaluated on this platform.")
ort_note = (f"The ONNX export reproduces the PyTorch outputs (max abs. difference {ort_maxdiff:.1e} on a verification batch) "
            f"with test accuracy {acc_ort:.4f}; ONNX Runtime CPU latency is {ort_lat[1]:.2f} ms at batch 1 and "
            f"{ort_lat[64]:.2f} ms at batch 64."
            if ORT_OK else "ONNX Runtime numbers were not measured on this platform.")
artifacts = [f"PyTorch bundle (sha256 {sha[:16]}...)"]
artifacts += ["torch.export program (.pt2)"] if EXPORT_OK else []
artifacts += ["TorchScript trace"] if TS_OK else []
artifacts += ["ONNX graph with a dynamic batch dimension"] if ONNX_OK else []

MODEL_CARD = f"""# Model card: mnist_cnn v1.0.0

## Model details
- Developed in Notebook 15 of the course "Deep Learning with PyTorch" as a teaching example.
- Architecture: two 3x3 convolutions (16 and 32 channels, ReLU, 2x2 max-pooling), a 128-unit hidden layer with dropout 0.25, and a 10-way linear output; {count_params(model):,} parameters, {size_kb:.0f} KB as a float32 bundle.
- Trained with Adam (lr 1e-3, cosine schedule) for {EPOCHS} epoch(s) on {N_TRAIN:,} MNIST training images, seed {SEED}. PyTorch {torch.__version__}.
- Inputs: float32 tensors of shape (N, 1, 28, 28) with pixel values in [0, 1]; the serving wrapper applies the normalization (mean {MNIST_MEAN}, std {MNIST_STD}). Outputs: probabilities over the digits 0-9.
- Artifacts: {'; '.join(artifacts)}.

## Intended use
- Classification of isolated, centred, white-on-black handwritten digits of the MNIST style. Intended as a course example and as a template for deployment work.
- Not intended for: multi-digit strings, printed digits, digits embedded in documents, or any application where a wrong digit has serious consequences without human review.

## Metrics and evaluation data
- Test accuracy on {N_TEST:,} MNIST test images: **{BASE_ACC:.4f}**. Per-class accuracy: {', '.join(f'{c}: {a:.3f}' for c, a in enumerate(pca))}. Weakest class: {worst}.
- Latency (PyTorch eager, this machine, {device.type}): {lat_rows[0][1]['p50']:.2f} ms at batch 1, {lat_rows[3][1]['p50']:.2f} ms at batch 64 (p50). {ort_note}
- {int8_note}

## Training data
- MNIST (LeCun et al., 1998): 60,000 training and 10,000 test images of size 28x28, digits written by US Census Bureau employees and high-school students; the class balance is approximately uniform.

## Factors and known limitations
- The model is sensitive to the input convention: inverted images (black digits on white) or unnormalized 0-255 inputs reduce accuracy to chance level; the monitoring statistics of Section 8.1 detect both.
- Rotations beyond about 15 degrees, thick strokes and non-centred digits are under-represented in the training data and degrade accuracy.
- The model has no notion of "not a digit": any 28x28 input receives a confident digit label. Use the confidence and an out-of-distribution score (Notebook 16 · Interpretability, Robustness and Uncertainty) to reject such inputs.

## Ethical considerations
- The data contains no personal information. Handwriting style varies with age, education and motor ability; per-group accuracy has not been evaluated and should be before any use that affects people.

## Caveats and recommendations
- Re-validate after any change of the input pipeline; monitor PSI of input intensity and prediction confidence; roll back to the previous registry version if PSI exceeds 0.25.
- Prefer the ONNX artifact for CPU serving and the int8 variant when memory-bound; verify numerics after every export.
"""
card_path = MODEL_DIR / "MODEL_CARD.md"
card_path.write_text(MODEL_CARD, encoding="utf-8")
print(f"wrote {card_path}\n")
print(MODEL_CARD)

The card is short by design. Its value lies in the sections that are uncomfortable to write, the
limitations and the intended-use boundaries, and in the fact that every number in it was measured by
the notebook that produced the model, so that regenerating the model regenerates the card.

> **Key idea.** A model in production is a process, not a file: monitored inputs and outputs, staged
> roll-outs with a rollback path, a reproducible pipeline, and documentation of what the model is for
> and where it fails.

## Exercises

### Exercise 1 (warm-up): export a ResNet to ONNX with a dynamic batch

Notebook 06 · Convolutional Neural Networks built a small ResNet for CIFAR-10 (3 × 32 × 32 inputs).
Define a compact version of it here
(a stem convolution, three stages of one `BasicBlock` each with widths 16/32/64 and stride-2
downsampling, global average pooling and a linear head; random weights are fine), export it to ONNX
with a dynamic batch dimension, and verify with ONNX Runtime that batches of 1 and 5 images produce the
same outputs as PyTorch. Which ONNX operators does BatchNorm turn into in evaluation mode?

**Solution to Exercise 1**

The export uses the same attempt list as Section 3.4 (dynamo exporter with a dynamic batch, then the
legacy exporter). Because the model is exported in `eval()` mode, BatchNorm appears either as a
`BatchNormalization` node with fixed statistics or, after the exporter's optimizations, folded into
the preceding `Conv`'s weights and bias; the residual additions become `Add`, and the global average
pool becomes `GlobalAveragePool` or a `ReduceMean`.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.shortcut = nn.Sequential()
        if stride != 1 or c_in != c_out:
            self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class TinyResNet(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU())
        self.layer1, self.layer2, self.layer3 = BasicBlock(16, 16), BasicBlock(16, 32, 2), BasicBlock(32, 64, 2)
        self.head = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.layer3(self.layer2(self.layer1(self.stem(x))))
        return self.head(F.adaptive_avg_pool2d(x, 1).flatten(1))


torch.manual_seed(SEED)
resnet = TinyResNet().eval()
x_res = torch.randn(4, 3, 32, 32)
res_path = MODEL_DIR / "tiny_resnet.onnx"
res_attempts = [
    ("dynamo, dynamic batch", lambda: torch.onnx.export(resnet, (x_res,), str(res_path), dynamo=True, external_data=False,
                                                       input_names=["image"], output_names=["logits"],
                                                       dynamic_shapes={"x": {0: torch.export.Dim("batch", min=2, max=1024)}})),
    ("legacy exporter", lambda: torch.onnx.export(resnet, (x_res,), str(res_path), dynamo=False, opset_version=17,
                                                 input_names=["image"], output_names=["logits"],
                                                 dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}})),
]
res_ok = False
for name, attempt in res_attempts:
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            attempt()
        res_ok = True
        print(f"exported TinyResNet with the {name} ({res_path.stat().st_size / 1024:.0f} KB)")
        break
    except Exception as e:
        print(f"{name} failed: {type(e).__name__}: {str(e)[:100]}")

if res_ok and HAVE_ORT:
    try:
        sess_res = ort.InferenceSession(str(res_path), providers=["CPUExecutionProvider"])
        name_in = sess_res.get_inputs()[0].name
        for bs in (1, 5):
            xb = torch.randn(bs, 3, 32, 32)
            with torch.inference_mode():
                ref = resnet(xb)
            out = sess_res.run(None, {name_in: xb.cpu().numpy().astype(np.float32)})[0]
            print(f"batch {bs}: max |ORT - PyTorch| = {np.abs(out - np.asarray(ref)).max():.2e}")
        if HAVE_ONNX:
            print("operators:", dict(Counter(n.op_type for n in onnx.load(str(res_path)).graph.node)))
    except Exception as e:
        print("ONNX Runtime check failed:", type(e).__name__, str(e)[:150])
else:
    print("skipping the runtime check (export failed or onnxruntime missing)")

Both batch sizes agree with PyTorch to about $10^{-6}$, including batch 1, which the ONNX graph accepts
because a symbolic dimension in ONNX carries no lower bound (the `torch.export` guard of Section 3.2
applies only to the PyTorch-side program). The operator list shows what became of the modules: `Conv`,
`Relu`, `Add` for the residual sums, a global pooling node and a `Gemm`; if `BatchNormalization` is
absent, the exporter folded it into the convolutions, which is exactly what a deployment runtime would
do anyway.

### Exercise 2: quantize the student and measure

Apply dynamic int8 quantization to the distilled MLP student from Section 6 and measure file size,
accuracy and CPU latency at batch sizes 1 and 64, as in Section 4.2. Compare the speed-up with the one
obtained for the CNN and explain the difference.

**Solution to Exercise 2**

The MLP consists only of linear layers, so after quantization *every* multiply-accumulate runs in
int8, and the model is now bandwidth-bound on a 784 × 64 matrix rather than compute-bound on
convolutions. Both size and latency therefore improve, in contrast with the CNN, where only the
parameter-heavy but compute-light linear layers changed.

In [ ]:
if DYNQ_OK:
    student_cpu = copy.deepcopy(student_kd).cpu().eval()
    student_q = tq.quantize_dynamic(copy.deepcopy(student_cpu), {nn.Linear}, dtype=torch.qint8)
    s_fp32, s_int8 = state_dict_bytes(student_cpu), state_dict_bytes(student_q)
    a_fp32 = evaluate(student_cpu, X_test.cpu(), y_test.cpu())[1]
    a_int8 = evaluate(student_q, X_test.cpu(), y_test.cpu())[1]
    print(f"student size: {s_fp32 / 1024:.1f} KB -> {s_int8 / 1024:.1f} KB ({s_fp32 / s_int8:.2f}x) | accuracy {a_fp32:.4f} -> {a_int8:.4f}")
    with torch.inference_mode():
        for bs in (1, 64):
            xb = X_test[:bs].cpu().contiguous()
            l32 = benchmark_latency(lambda: student_cpu(xb), n_warmup=5, n_iters=40)["p50"]
            l8 = benchmark_latency(lambda: student_q(xb), n_warmup=5, n_iters=40)["p50"]
            print(f"batch {bs:3d}: fp32 {l32:.3f} ms | int8 {l8:.3f} ms | speed-up {l32 / l8:.2f}x "
                  f"(CNN at the same batch: {dyn_lat[('fp32', bs)] / dyn_lat[('int8', bs)]:.2f}x)")
else:
    print("dynamic quantization is unavailable on this platform; see Section 4.2.")

### Exercise 3: iterative magnitude pruning with fine-tuning

Section 5 pruned in one shot. Implement iterative magnitude pruning: reach 95 % global sparsity in
three rounds (50 %, 80 %, 95 %), fine-tuning for one epoch after each round, and compare the final
accuracy with one-shot pruning to 95 % followed by the same total amount of fine-tuning (three
epochs). Remember that `prune.global_unstructured` applied repeatedly interprets `amount` relative to
the weights that are still unpruned.

**Solution to Exercise 3**

To move from a current sparsity $s_{\text{cur}}$ to a target $s_{\text{new}}$ we must remove the
fraction $(s_{\text{new}} - s_{\text{cur}}) / (1 - s_{\text{cur}})$ of the *remaining* weights. The
iterative schedule lets the network re-adapt between rounds, so the weights removed in later rounds
are chosen from a network that has already compensated for the earlier removals, which is why it
usually ends up more accurate than one-shot pruning with the same fine-tuning budget (Han et al.,
2015; the same idea underlies the lottery-ticket experiments of Frankle and Carbin, 2019).

In [ ]:
N_FT = 4_000 if FAST_RUN else 60_000
X_ft, y_ft = X_train[:N_FT], y_train[:N_FT]


def fine_tune(m, epochs, lr=3e-4):
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    for _ in range(epochs):
        train_one_epoch(m, opt, X_ft, y_ft)
    return m


targets = [0.5, 0.8, 0.95]
iterative = copy.deepcopy(model)
current = 0.0
for target in targets:
    rel = (target - current) / (1 - current)
    prune.global_unstructured(prunable(iterative), pruning_method=prune.L1Unstructured, amount=rel)
    current = total_sparsity(iterative)
    fine_tune(iterative, 1)
    print(f"round -> target {target:.0%}: pruned {rel:.0%} of the remaining weights, sparsity now {current:.3f}, "
          f"acc after fine-tune {evaluate(iterative, X_test, y_test)[1]:.4f}")

one_shot = prune_global_copy(model, 0.95)
fine_tune(one_shot, len(targets))
print(f"\none-shot 95% + {len(targets)} fine-tune epochs: acc {evaluate(one_shot, X_test, y_test)[1]:.4f} (sparsity {total_sparsity(one_shot):.3f})")
print(f"iterative 50/80/95% + 1 epoch each  : acc {evaluate(iterative, X_test, y_test)[1]:.4f} (sparsity {total_sparsity(iterative):.3f})")

Both schedules end at the same sparsity with the same fine-tuning budget; the iterative one is usually
a little more accurate, and the gap widens at more extreme sparsity or with a bigger model. In
practice the schedule is extended over many rounds with a smooth increase of the sparsity (the "cubic"
schedule of Zhu and Gupta, 2017) and is combined with the learning-rate schedule of the original
training.

### Exercise 4: distillation temperature

Repeat the distillation of Section 6 with temperatures $T \in \{1, 2, 4, 8\}$ (keeping
$\alpha = 0.7$) and plot the final student accuracy against $T$. Is $T = 1$ (matching the teacher's
raw probabilities) already better than hard labels? Comment on the reliability of your conclusion.

**Solution to Exercise 4**

`train_student` accepts the temperature directly. Since all runs share the seed and the batch order,
the differences between temperatures are not confounded by initialization or data order; they may
still be within the run-to-run noise of a single seed, which is why the honest conclusion requires
several seeds. Typically $T = 1$ already helps (the teacher's probabilities are more informative than
one-hot labels), moderate temperatures of 2–5 help a little more, and very high temperatures flatten
the targets so much that the soft term carries little information.

In [ ]:
temperatures = [1, 2, 4, 8]
temp_acc = []
for T in temperatures:
    _, hist_T = train_student(distill=True, epochs=EPOCHS_KD, T=float(T))
    temp_acc.append(hist_T[-1])
    print(f"T = {T}: final test acc {hist_T[-1]:.4f}")
print(f"hard labels only: {hist_scratch[-1]:.4f}")

plt.figure()
plt.plot(temperatures, [a * 100 for a in temp_acc], "o-", label="distilled student")
plt.axhline(hist_scratch[-1] * 100, color="C3", ls="--", label="student, hard labels")
plt.xscale("log", base=2); plt.xticks(temperatures, [str(t) for t in temperatures])
plt.xlabel("temperature T"); plt.ylabel("test accuracy (%)"); plt.title("Effect of the distillation temperature")
plt.legend(); plt.tight_layout(); plt.show()

### Exercise 5 (challenge): a request-batching simulation

Write a discrete-event simulation of a server with dynamic batching: requests arrive as a Poisson
process with rate $\lambda$; the server collects requests until either `max_batch` are waiting or
`max_wait` milliseconds have passed since it became free, then runs the batch, whose duration follows
the linear model $\ell(b) = a + c\,b$ fitted to the latencies measured in Section 1.3. Report
throughput and mean and p99 latency as a function of `max_batch` for a load below and a load above the
capacity of unbatched serving ($1/\ell(1) = 1/(a + c) \approx 1/a$ requests per millisecond), and
interpret.

**Solution to Exercise 5**

The simulation keeps one clock: the time at which the server becomes free. From there it collects the
next batch: if `max_batch` requests are already waiting, the batch starts immediately; otherwise the
server waits up to `max_wait` for more arrivals and then launches whatever it has. Each request's
latency is its completion time minus its arrival time, which includes queueing, the batching wait and
the batch's compute time. The linear latency model comes from a least-squares fit of the measured
p50 latencies.

In [ ]:
bs_meas = np.array([bs for bs, _ in lat_rows], dtype=float)
lat_meas = np.array([st["p50"] for _, st in lat_rows])
c_slope, a_intercept = np.polyfit(bs_meas, lat_meas, 1)
a_intercept = max(a_intercept, 1e-3)
latency_model = lambda b: a_intercept + c_slope * b
print(f"latency model: {a_intercept:.3f} ms + {c_slope:.4f} ms x batch  (unbatched capacity ~{1000 / a_intercept:,.0f} req/s)")


def simulate_batching(rate_per_ms, max_batch, max_wait_ms, latency_fn, n_requests=4000, seed=0):
    rng = np.random.default_rng(seed)
    arrivals = np.cumsum(rng.exponential(1.0 / rate_per_ms, n_requests))
    t_free, i, latencies = 0.0, 0, []
    while i < n_requests:
        start = max(t_free, arrivals[i])                  # server is free; the first waiting request defines the batch start
        deadline = start + max_wait_ms
        j = i + 1
        while j < n_requests and j - i < max_batch and arrivals[j] <= deadline:
            j += 1
        full = (j - i == max_batch)
        launch = max(start, arrivals[j - 1]) if full else deadline
        finish = launch + latency_fn(j - i)
        latencies.extend((finish - arrivals[i:j]).tolist())
        t_free, i = finish, j
    lat = np.array(latencies)
    return {"throughput": n_requests / (t_free - arrivals[0]) * 1000.0, "mean": lat.mean(), "p99": np.percentile(lat, 99)}


loads = {"low load (0.5x unbatched capacity)": 0.5 / a_intercept, "high load (3x unbatched capacity)": 3.0 / a_intercept}
max_batches = [1, 2, 4, 8, 16, 32, 64]
results = {}
for label, rate in loads.items():
    results[label] = [simulate_batching(rate, mb, max_wait_ms=2.0, latency_fn=latency_model) for mb in max_batches]
    print(f"\n{label}: {rate * 1000:,.0f} req/s")
    for mb, r in zip(max_batches, results[label]):
        print(f"  max_batch {mb:3d}: throughput {r['throughput']:9,.0f} req/s | mean latency {r['mean']:9.2f} ms | p99 {r['p99']:9.2f} ms")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, rs in results.items():
    axes[0].plot(max_batches, [r["p99"] for r in rs], "o-", label=label)
    axes[1].plot(max_batches, [r["throughput"] for r in rs], "o-", label=label)
for ax, ylabel, title in [(axes[0], "p99 latency (ms)", "Latency vs batching"), (axes[1], "throughput (req/s)", "Throughput vs batching")]:
    ax.set_xscale("log", base=2); ax.set_xlabel("max_batch"); ax.set_ylabel(ylabel); ax.set_title(title); ax.legend(fontsize=8)
axes[0].set_yscale("log")
plt.tight_layout(); plt.show()

At low load, batching *hurts*: the server is idle most of the time, and every request pays the
`max_wait` for companions that rarely arrive, so `max_batch = 1` (or a `max_wait` of zero) gives the
best latency. Above the unbatched capacity, `max_batch = 1` is catastrophic, the queue grows without
bound and the p99 latency explodes, while larger batches bring the server back under the load with a
p99 of a few milliseconds. A production batcher therefore adapts: it launches without waiting when
only a few requests are queued and batches aggressively when many are, which is what the "collect
until full or until the deadline" rule approximates once `max_wait` is small relative to the batch
compute time (a queue that is already longer than `max_batch` fills a batch instantly, while a nearly
empty queue costs at most `max_wait` of extra latency).

## Summary

- Inference differs from training in mode (`eval()`), autograd (`inference_mode`), batching and
  metrics: report p50/p99 latencies over batch sizes and serve near the knee of the throughput curve.
- Persist `state_dict`s and metadata, not pickled modules; version models semantically, keep the
  preprocessing inside the artifact, hash the file, and track versions in a registry.
- `torch.export` produces a sound, guarded ATen graph (the basis of AOTInductor, ExecuTorch and the
  ONNX exporter); TorchScript tracing bakes in control flow and is in maintenance; ONNX plus ONNX
  Runtime gives a Python-free, often faster CPU deployment. Always verify exports numerically at
  more than one batch size.
- Affine quantization maps floats to integers with a scale and zero-point; per-channel int8 is nearly
  lossless, 4 bits and below need quantization-aware training with the straight-through estimator.
  Dynamic quantization shrinks linear layers; static quantization is what speeds up convolutions.
- Magnitude pruning removes 80–95 % of weights with little loss after fine-tuning, but unstructured
  sparsity does not reduce latency on dense hardware; structured pruning and distillation do.
- Knowledge distillation trains a small student on the teacher's softened distribution
  ($T^2$-scaled KL plus cross-entropy) and beats training from scratch at the same size.
- A service loads the model once, validates inputs with a schema, exposes health and version, and
  batches requests; Triton, TorchServe, vLLM and edge runtimes industrialize this.
- Operate the model as a process: monitor PSI of inputs and outputs, roll out through shadow, canary
  and A/B stages, keep rollbacks a registry operation, and document the model in a card.

## Further reading

- Hinton, Vinyals and Dean, 2015, *Distilling the Knowledge in a Neural Network* — soft targets and temperature.
- Han, Pool, Tran and Dally, 2015, *Learning both Weights and Connections for Efficient Neural Networks* — iterative magnitude pruning; Zhu and Gupta, 2017, *To Prune, or Not to Prune*; Frankle and Carbin, 2019, *The Lottery Ticket Hypothesis*.
- Jacob et al., 2018, *Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference* (CVPR) — the affine int8 scheme and QAT; Nagel et al., 2021, *A White Paper on Neural Network Quantization*; Gholami et al., 2021, *A Survey of Quantization Methods for Efficient Neural Network Inference*.
- Mitchell et al., 2019, *Model Cards for Model Reporting* (FAT*).
- Sculley et al., 2015, *Hidden Technical Debt in Machine Learning Systems* (NeurIPS) — why the model is the smallest part of the system.
- Crankshaw et al., 2017, *Clipper: A Low-Latency Online Prediction Serving System* (NSDI) — adaptive batching and model selection in a serving layer.
- PyTorch documentation: `torch.export` (https://docs.pytorch.org/docs/stable/export.html), ONNX export (https://docs.pytorch.org/docs/stable/onnx.html), quantization (https://docs.pytorch.org/docs/stable/quantization.html), and the pruning tutorial in the official tutorials.
- ONNX Runtime documentation (https://onnxruntime.ai/docs/), in particular the graph optimization and quantization guides.